# Sandbox

Restates the model from `04_dynamic_hat_algebra.ipynb` (parameters, temporary equilibrium, dynamic hat algebra, transition-path solver) -- see that notebook for the full derivation and solver mechanics -- then spends the rest of this notebook exploring shock scenarios and parameter variations, which is what's unique here. Self-contained: does not need 04 to be run first.

In [1]:
import Pkg; Pkg.add(["Ipopt", "SpecialFunctions", "DataFrames"])


    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


In [2]:
using Random, Distributions, JuMP, Ipopt, SpecialFunctions, DataFrames


# Parameters

This section sets the model's fundamentals and structural parameters, and fixes the indexing convention used throughout the rest of the notebook.

**Regions and markets.** There are $N$ regions and $J$ real sectors, plus a non-employment "sector 0" in every region, giving $M = J+1$ markets per region. Any array indexed over every possible market a household could occupy (labor $L$, value function $V$, migration shares $\mu$, mobility costs $\tau$) is `N × M`, with **column 1 = non-employment** and **columns 2:M = real sectors 1..J**. Arrays that only pertain to production ($A$, $w$, $\kappa$, $\theta$, $\eta$, $\gamma$, $\alpha$) are `N × J`, since sector 0 has no production, wage, or price — to compare a market-indexed array against a sector-indexed one, offset the sector index by $+1$.

**Time-varying fundamentals**, $\Theta_t = (A_t, \kappa_t)$:
- $A_t^{nj}$ — productivity in region $n$, sector $j$
- $\kappa_t^{nj,ij}$ — iceberg trade cost shipping sector-$j$ goods from region $i$ to region $n$

**Constant fundamentals**, $\bar\Theta = (\Upsilon, b)$:
- $\Upsilon = \{\tau^{nj,ik}\}$ — labor relocation (mobility) costs, in utility terms, from market $(n,j)$ to market $(i,k)$
- $b^n$ — value of home production in region $n$ (a non-employed household's consumption)

**Structural parameters:**
- $\beta \in [0,1)$ — discount factor
- $\theta^j$ — Fréchet trade elasticity in sector $j$
- $\nu$ — dispersion of the idiosyncratic migration taste shock ($1/\nu$ is the migration elasticity)
- $\alpha^j$ — Cobb-Douglas consumption share on sector $j$, with $\sum_j \alpha^j = 1$

**State variable:** $L_t = \{L_t^{nj}\}$, the mass of households in each market at time $t$ — the only object carrying information from one period to the next.

In [3]:
N = 2 # Number of regions
J = 3 # Number of real sectors (sector 0 / non-employment is handled separately -- see M below)
M = J + 1 # Number of markets per region: non-employment (market column 1) plus the J real
          # sectors (market columns 2:M). Arrays indexed over every market a household could be
          # in (L, V, mu, tau_mig) are N x M, with column 1 = non-employment. Arrays that only
          # pertain to production (A, w, kappa, theta, eta, gamma, alpha) are N x J -- to compare
          # a market-indexed array against a sector-indexed one, offset the sector index by +1.
n_omega = 10000 # number of varieties used to discretize the omega in [0,1] continuum per region-sector


L_0  = ones(N,M) # labor force at time 0, over all N regions x M markets (incl. non-employment)

Random.seed!(1) # fixed seed for reproducibility

A_0 = rand(N,J) # region-sector productivity (real sectors only; undefined for non-employment)

B = ones(N,J) # constant coefficient in the unit-cost function

w_0 = ones(N,J) # wage, real sectors only -- non-employment pays no wage; households there consume b_n instead

b = ones(N) # value of home production: consumption of a non-employed household in region n

kappa_0 = [ones(N,N) for _ in 1:J] # iceberg trade costs, per sector

theta = fill(4.0, J) # Frechet shape parameter per sector (governs dispersion of productivity draws
                      # across the continuum of varieties/regions, and the trade elasticity);
                      # scale is absorbed into A_0, so no separate T parameter is needed

eta = fill(2.0, N, J) # elasticity of substitution across varieties within sector j (CES aggregator)

gamma = ones(N, J) # labor/value-added share of production; = 1 since this simplified model has no materials

beta = 0.95 # household discount factor
# utility cost of moving from market (n,j) to market (i,k); 0 to stay in the same market, 1 otherwise
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J) # Cobb-Douglas consumption shares: alpha[g] is the share of household income
alpha = alpha ./ sum(alpha) # spent on good g, common across all regions/sectors; normalized to sum to 1

nu = 1.0 # dispersion of the idiosyncratic (Frechet/type-I EV) taste shock over migration destinations
         # (larger nu = migration is less sensitive to utility differences, i.e. more friction)

T = 10 # number of periods to simulate the labor dynamics forward


10

## Baseline levels

Everything from here on is expressed in *changes* relative to a baseline allocation -- so we
first need an actual baseline. We solve the Temporary Equilibrium (Definition 1)
at $L_0$ to get market-clearing wages $w_{temp}$ and trade shares $\pi_{temp}$; these play the
role of $(w_t, \pi_t)$ in the hat-algebra equations below.

In [4]:
# Temporary equilibrium (Definition 1): given labor supply L and the fundamentals (A_0, kappa_0),
# find wages w = {w^{nj}} that clear goods and labor markets simultaneously -- eq 6 (goods
# clearing / expenditure) and eq 7 (labor clearing) together, N x J equations in N x J wage
# unknowns. Solved via JuMP + Ipopt as a feasibility problem: satisfy the market-clearing
# equations as constraints, with no objective to minimize.
#
# The system is homogeneous of degree 1 in w: scaling every wage by a constant lambda scales the
# wage bill (LHS of eq 7) by lambda, and scales expenditure X (eq 6) by lambda too, while trade
# shares pi (eq 5) are unaffected (they only depend on RELATIVE wages across sources). So only
# relative wages are pinned down -- we normalize w[1,1] = 1 and drop that market's own clearing
# equation, which is redundant with the rest by Walras' law (aggregate labor income always equals
# aggregate expenditure here, since there is no trade deficit).

function trade_shares_and_expenditure(w::AbstractMatrix, L::AbstractMatrix)
    x = B .* w # unit cost (eq 4.2)
    trade_cost_term = [
        (x[i,j] * kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    pi = [trade_cost_term[n,j,i] / sum(trade_cost_term[n,j,:]) for n in 1:N, j in 1:J, i in 1:N] # eq 5
    I = vec(sum(w .* L[:, 2:M], dims=2)) # labor income by region (employed markets only, eq analog of I_0 above)
    X = I * alpha' # eq 6
    return pi, X
end

function solve_temporary_equilibrium(L::AbstractMatrix; w_guess = ones(N,J))
    model = Model(Ipopt.Optimizer)
    set_silent(model)
    @variable(model, w[n=1:N, j=1:J] >= 1e-6, start = w_guess[n,j])

    @expression(model, tct[n=1:N, j=1:J, i=1:N],
        (B[i,j]*w[i,j]*kappa_0[j][n,i])^(-theta[j]) * A_0[i,j]^(theta[j]*gamma[i,j]))
    @expression(model, pishare[n=1:N, j=1:J, i=1:N], tct[n,j,i] / sum(tct[n,j,m] for m in 1:N))
    @expression(model, Inc[n=1:N], sum(w[n,k]*L[n,k+1] for k in 1:J))
    @expression(model, X[n=1:N, j=1:J], alpha[j]*Inc[n])

    @constraint(model, w[1,1] == 1.0) # numeraire, replaces the redundant (1,1) equation
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w[n,j]*L[n,j+1] == sum(pishare[i,j,n]*X[i,j] for i in 1:N)) # eq 7
        end
    end

    @objective(model, Min, 0) # feasibility problem: no objective, just satisfy the constraints
    optimize!(model)

    w_star = value.(w)
    pi_star, X_star = trade_shares_and_expenditure(w_star, L)
    return w_star, pi_star, X_star, termination_status(model)
end


solve_temporary_equilibrium (generic function with 1 method)

In [5]:
# Solve the temporary equilibrium at the initial labor distribution L_0 -- this is the
# baseline (w_temp, pi_temp) that the hat-algebra system below is defined relative to.

w_temp, pi_temp, X_temp, status_temp = solve_temporary_equilibrium(L_0)
println("Solver status: ", status_temp)



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

Solver status: LOCALLY_SOLVED


In [6]:
# stationary_V(U_mkt): the Bellman fixed point that solves for each market's value function,
# given flow utility U_mkt -- needed to get the baseline mu_stationary below (the migration
# shares of the economy's own stationary equilibrium under w_temp).
function stationary_V(U_mkt::AbstractMatrix; tol=1e-12, maxiter=10_000)
    V = log.(U_mkt)
    for _ in 1:maxiter
        V_next = [
            log(U_mkt[n,j]) + nu * log(sum(exp((beta*V[i,k] - tau_mig[n,j,i,k]) / nu) for i in 1:N, k in 1:M))
            for n in 1:N, j in 1:M
        ]
        if maximum(abs.(V_next .- V)) < tol
            V = V_next
            break
        end
        V = V_next
    end
    return V
end

# migration_shares(V_next): the logit migration shares (eq 3), as a function of next period's
# value function.
function migration_shares(V_next::AbstractMatrix)
    [
        exp((beta*V_next[i,k] - tau_mig[n,j,i,k]) / nu) /
        sum(exp((beta*V_next[m,h] - tau_mig[n,j,m,h]) / nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ]
end


migration_shares (generic function with 1 method)

# Dynamic Hat Algebra

Everything above is written in terms of *levels* (actual values of $A$, $\kappa$, $w$). This section rewrites the same two blocks -- the production side and the household's migration decision -- in terms of *proportional changes* relative to those levels instead, since a counterfactual is usually given as a change ("productivity up 20%") rather than a new absolute level. In the code, every such ratio is named with a `_dot` suffix (`w_dot`, `A_dot`, `L_dot`, `u_dot`, ...).

Mechanically, the two functions defined in this section (`solve_temp_eq_hat` and `stationary_u_hat`/`migration_shares_next`) work exactly like `solve_temporary_equilibrium` and `stationary_V`/`migration_shares` above -- same JuMP/Ipopt solve, same fixed-point iteration -- just applied to the ratio versions of the equations. They're the building blocks that `solve_transition_path_hat` (last section of the model setup) calls once per period to trace out a full transition path, and that every scenario in the rest of this notebook calls through that solver.

## Temporary equilibrium in time differences

Ratio-form version of the Baseline Levels solve above: given a baseline allocation $(w_t, \pi_t, L_t)$ and a change in labor supply $\dot L_{t+1}=L_{t+1}/L_t$ and fundamentals $\dot A_{t+1}, \dot\kappa_{t+1}$, solves for the wage change $\dot w_{t+1}=w_{t+1}/w_t$ -- without ever needing the level of $A_t$ or $\kappa_t$, only their proportional change.

$$\dot x_{t+1}^{nj} = \dot w_{t+1}^{nj} \tag{8}$$
$$\dot P_{t+1}^{nj} = \left(\sum_{i=1}^N \pi_t^{nj,ij}\left(\dot w_{t+1}^{ij}\dot\kappa_{t+1}^{nj,ij}\right)^{-\theta^j}\left(\dot A_{t+1}^{ij}\right)^{\theta^j}\right)^{-1/\theta^j} \tag{9}$$
$$\pi_{t+1}^{nj,ij} = \pi_t^{nj,ij}\left(\frac{\dot w_{t+1}^{ij}\dot\kappa_{t+1}^{nj,ij}}{\dot P_{t+1}^{nj}}\right)^{-\theta^j}\left(\dot A_{t+1}^{ij}\right)^{\theta^j} \tag{10}$$
$$X_{t+1}^{nj} = \alpha^j\sum_{k=1}^J \dot w_{t+1}^{nk}\dot L_{t+1}^{nk}\, w_t^{nk}L_t^{nk} \tag{11}$$
$$\dot w_{t+1}^{nj}\dot L_{t+1}^{nj}\, w_t^{nj}L_t^{nj} = \sum_{i=1}^N \pi_{t+1}^{ij,nj} X_{t+1}^{ij} \tag{12}$$

`solve_temp_eq_hat` (below) is coded exactly like `solve_temporary_equilibrium` above: it declares `w_dot` as unknowns in a JuMP model, writes eqs. (9)-(11) as plain formulas in terms of `w_dot`, writes eq. (12) as one constraint per market, and calls Ipopt to find a `w_dot` that satisfies all of them. Eq. (10) already multiplies in the baseline $\pi_t$, so `pi_next` comes out as the actual level $\pi_{t+1}$ (not a further ratio) -- exactly what eq. (12) needs, since it's a level equation. As with the levels version, only relative wages are pinned down, so we again normalize $\dot w_{t+1}^{11}=1$ and drop that market's own equation.

In [7]:
# Temporary equilibrium in time differences (eqs 8-12): given a baseline allocation at t
# (pi_t, w_t, L_t) and a change in labor supply / fundamentals (L_dot, A_dot, kappa_dot), solve
# for w_dot = w_{t+1}/w_t. kappa_dot has the same shape as kappa_0: a length-J vector of N x N
# matrices. L_dot is N x M (market space, column 1 = non-employment); only its real-sector
# columns (2:M) enter here, offset by +1 as usual.
#
# pi_next (eq 10) and X_next (eq 11) come out as LEVELS despite living inside the hat system --
# pi_next already has the baseline pi_t multiplied in, and X_next is built from t-level income
# terms -- so eq 12 (a level equation: nominal revenue at t+1 on both sides) uses them directly,
# with no further multiplication by pi_t needed.

function solve_temp_eq_hat(pi_t::Array{Float64,3}, w_t::AbstractMatrix, L_t::AbstractMatrix,
                            L_dot::AbstractMatrix, A_dot::AbstractMatrix, kappa_dot;
                            w_dot_guess = ones(N,J))
    model = Model(Ipopt.Optimizer)
    set_silent(model)
    @variable(model, w_dot[n=1:N, j=1:J] >= 1e-6, start = w_dot_guess[n,j])

    @expression(model, P_dot[n=1:N, j=1:J],
        (sum(pi_t[n,j,i] * (w_dot[i,j]*kappa_dot[j][n,i])^(-theta[j]) * A_dot[i,j]^theta[j] for i in 1:N))^(-1/theta[j])) # eq 9
    @expression(model, pi_next[n=1:N, j=1:J, i=1:N],
        pi_t[n,j,i] * ((w_dot[i,j]*kappa_dot[j][n,i]) / P_dot[n,j])^(-theta[j]) * A_dot[i,j]^theta[j]) # eq 10
    @expression(model, X_next[n=1:N, j=1:J],
        alpha[j] * sum(w_dot[n,k]*L_dot[n,k+1]*w_t[n,k]*L_t[n,k+1] for k in 1:J)) # eq 11

    @constraint(model, w_dot[1,1] == 1.0) # numeraire
    for n in 1:N, j in 1:J
        if !(n == 1 && j == 1)
            @constraint(model, w_dot[n,j]*L_dot[n,j+1]*w_t[n,j]*L_t[n,j+1] == sum(pi_next[i,j,n]*X_next[i,j] for i in 1:N)) # eq 12
        end
    end

    @objective(model, Min, 0)
    optimize!(model)

    return value.(w_dot), value.(P_dot), value.(pi_next), value.(X_next), termination_status(model)
end


solve_temp_eq_hat (generic function with 1 method)

## Sequential equilibrium in time differences (household block)

Ratio-form version of the household migration decision: $\mu_t^{nj,ik}$ is the share of households in market $(n,j)$ who move to market $(i,k)$ next period, and $u_t^{nj}\equiv\exp(V_t^{nj})$.

$$\mu_{t+1}^{nj,ik} = \frac{\mu_t^{nj,ik}\left(\dot u_{t+2}^{ik}\right)^{\beta/\nu}}{\sum_{m=1}^N\sum_{h=0}^J \mu_t^{nj,mh}\left(\dot u_{t+2}^{mh}\right)^{\beta/\nu}} \tag{13}$$
$$\dot u_{t+1}^{nj} = \dot\omega^{nj}(\dot L_{t+1},\dot\Theta_{t+1})\left(\sum_{i=1}^N\sum_{k=0}^J \mu_t^{nj,ik}\left(\dot u_{t+2}^{ik}\right)^{\beta/\nu}\right)^{\nu} \tag{14}$$
$$L_{t+1}^{nj} = \sum_{i=1}^N\sum_{k=0}^J \mu_t^{ik,nj}L_t^{ik} \tag{15}$$

$\dot\omega_{t+1}^{nj}=\dot w_{t+1}^{nj}/\dot P_{t+1}^n$ (real wage change) connects this block to the production block above -- `solve_temp_eq_hat`'s output feeds eq. 14 directly. Non-employment pays no wage, so $\dot\omega_{t+1}^{n0}=1$ always.

`stationary_u_hat` (below) specializes to the **stationary case**: a shock that permanently shifts $\dot\omega$ to a new constant value, so $\dot u_{t+1}=\dot u_{t+2}=\dot u^*$ for every $t$. It's coded exactly like `stationary_V` above -- starts from a guess (`u_dot = 1` everywhere), repeatedly recomputes eq. 14's right-hand side, and stops once the guess stops changing by more than `tol` -- just multiplying instead of adding at each step. `migration_shares_next` then applies eq. 13 directly to an already-solved `u_dot`: a single closed-form calculation, not a loop.

In [8]:
# Household block in time differences, stationary case (eqs 13-15). u_dot is the multiplicative
# analogue of the V fixed point above: u_dot^{nj} = omega_dot^{nj} *
# (sum_{i,k} mu_baseline^{nj,ik} * (u_dot^{ik})^{beta/nu})^nu. mu_baseline is the origin period's
# (level) migration shares, used as fixed weights, exactly as pi_t was used as fixed weights in
# P_dot (eq 9) above.

function stationary_u_hat(omega_dot::AbstractMatrix, mu_baseline::Array{Float64,4}; tol=1e-12, maxiter=10_000)
    u_dot = ones(N, M)
    for _ in 1:maxiter
        u_dot_next = [
            omega_dot[n,j] * (sum(mu_baseline[n,j,i,k] * u_dot[i,k]^(beta/nu) for i in 1:N, k in 1:M))^nu
            for n in 1:N, j in 1:M
        ]
        converged = maximum(abs.(u_dot_next .- u_dot)) < tol
        u_dot = u_dot_next
        converged && break
    end
    return u_dot
end

# eq 13: mu_{t+1} is a LEVEL despite living in the hat system (mu_baseline is already multiplied
# in), exactly parallel to pi_next above.
function migration_shares_next(u_dot::AbstractMatrix, mu_baseline::Array{Float64,4})
    [
        mu_baseline[n,j,i,k] * u_dot[i,k]^(beta/nu) /
        sum(mu_baseline[n,j,m,h] * u_dot[m,h]^(beta/nu) for m in 1:N, h in 1:M)
        for n in 1:N, j in 1:M, i in 1:N, k in 1:M
    ]
end

# flow_utility_mkt_at: computes flow utility at an explicit fundamentals level, rather than
# reading the global A_0/kappa_0 -- needed to get a baseline mu under the market-clearing wage
# w_temp rather than the placeholder w_0, in the Sandbox section below.

function flow_utility_mkt_at(w::AbstractMatrix, A::AbstractMatrix, kappa)
    x = B .* w
    trade_cost_term = [
        (x[i,j] * kappa[j][n,i])^(-theta[j]) * A[i,j]^(theta[j]*gamma[i,j])
        for n in 1:N, j in 1:J, i in 1:N
    ]
    Gamma = [SpecialFunctions.gamma((theta[j] + 1 - eta[n,j]) / theta[j])^(1 / (1 - eta[n,j])) for n in 1:N, j in 1:J]
    P = [Gamma[n,j] * sum(trade_cost_term[n,j,:])^(-1/theta[j]) for n in 1:N, j in 1:J]
    P_hat_region = [prod((P[n,g] / alpha[g])^alpha[g] for g in 1:J) for n in 1:N]
    C = [w[n,j] / P_hat_region[n] for n in 1:N, j in 1:J]
    return hcat(b, C)
end


flow_utility_mkt_at (generic function with 1 method)

# Solving for the Transition Path

Given a baseline allocation $(L_0,\pi_0,w_0)$, last period's migration shares $\mu_{-1}$, and a path of fundamental changes $\dot\Theta_t=(\dot A_t,\dot\kappa_t)$ for $t=1,\dots,T$ (no further change assumed after $T$), `solve_transition_path_hat` (below) computes the whole transition path $\{L_t,\mu_t,w_t\}$ by chaining the two blocks above together period by period.

It reuses eqs 8-15 exactly as defined above -- no new equations here, just a loop that calls `solve_temp_eq_hat` and applies eqs 13-15 repeatedly.

**Why there's an outer loop.** The migration-share path $\mu_t$ needs $\dot u_{t+1}$ to be simulated forward (eq. 13), but $\dot u_t$ needs the full $\mu$ path to be solved backward (eq. 14) -- neither can be computed first. The code handles this by *guessing* a path for $\dot u_t$, running one complete forward-then-backward pass to get an updated path, and repeating until the guess and the result agree. One pass through the outer loop does:

1. **Forward, using the current guess:** simulate $\mu_0,...,\mu_T$ (eq. 13) from $\mu_{-1}$, then simulate $L_1,...,L_{T+1}$ (eq. 15) from $L_0$ using that $\mu$ path.
2. **Forward again, solving the production side:** loop over $t=0,\dots,T$ a second time, calling `solve_temp_eq_hat` once per period against that period's running wage/trade-share levels, then updating those levels before moving to $t+1$. Levels (not just growth rates) have to be carried through this loop because eq. (11)'s $X_{t+1}$ needs the actual wage-bill level $w_t L_t$, which compounds period over period.
3. **Backward:** loop over $t=T-1,\dots,0$, applying eq. 14 starting from the fixed terminal condition $\dot u_{T+1}=1$, since $\dot u_{t+1}$ depends on the *next* period's $\dot u_{t+2}$. This produces an updated guess for the whole $\dot u$ path.
4. **Compare and update:** measure the largest change anywhere between the updated path and the guess used to run steps 1-2, then set the next guess to a weighted average of the two (see the `damp` comment in the code below for why). Repeat from step 1 until that largest change is below `tol`, or `max_outer` passes are used up.

In [9]:
# Indexing convention used throughout (Julia is 1-indexed, but the model's time index starts at
# 0 or -1 for several objects):
#   L_path[s]         = L_{s-1}      for s = 1,...,T+2   (L_path[1] = L_0, ..., L_path[T+2] = L_{T+1})
#   mu_path[s]         = mu_{s-1}     for s = 1,...,T+1   (mu_path[1] = mu_0, ..., mu_path[T+1] = mu_T)
#   omega_dot_path[s]  = omega_dot_s  for s = 1,...,T+1   (direct indexing, no shift)
#   u_dot_guess[s]     = u_dot_s      for s = 1,...,T     (direct indexing; u_dot_{T+1} = 1 is a
#                                                           separate fixed constant, not in this array)
#   A_dot_path[s], kappa_dot_path[s] = shock going INTO period s, i.e. Theta_dot_s = Theta_s/Theta_{s-1},
#                                      for s = 1,...,T (beyond T, Theta_dot is implicitly 1: no further change)

function solve_transition_path_hat(w_0::AbstractMatrix, L_0::AbstractMatrix, pi_0::Array{Float64,3},
                                    mu_minus1::Array{Float64,4}, A_dot_path::Vector, kappa_dot_path::Vector;
                                    T::Int, max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5)

    u_dot_guess = [ones(N,M) for _ in 1:T] # step 1: initial "no change" guess for u_dot_1,...,u_dot_T
    u_dot_terminal = ones(N,M)             # fixed terminal condition, u_dot_{T+1} = 1

    local mu_path, L_path, omega_dot_path, w_path, pi_final, converged, outer_used

    for outer in 1:max_outer
        # Step 2: forward-simulate mu_0,...,mu_T (eq 13), seeded by mu_minus1.
        # mu_t needs mu_{t-1} and u_dot_{t+1} -- for t=T that's the fixed terminal u_dot_{T+1}.
        mu_path = Vector{Array{Float64,4}}(undef, T+1)
        mu_prev = mu_minus1
        for t in 0:T
            u_dot_tp1 = (t+1 <= T) ? u_dot_guess[t+1] : u_dot_terminal
            mu_path[t+1] = migration_shares_next(u_dot_tp1, mu_prev)
            mu_prev = mu_path[t+1]
        end

        # Step 3: forward-simulate L_1,...,L_{T+1} (eq 15) -- one period past T, since mu_T
        # generates L_{T+1}, needed by step 4's last temporary-equilibrium solve (T -> T+1).
        L_path = Vector{Matrix{Float64}}(undef, T+2)
        L_path[1] = L_0
        for t in 0:T
            L_path[t+2] = [
                sum(mu_path[t+1][i,k,n,j] * L_path[t+1][i,k] for i in 1:N, k in 1:M)
                for n in 1:N, j in 1:M
            ]
        end

        # Step 4: solve the temporary equilibrium period by period, t=0,...,T, tracking the
        # LEVELS (pi_t, w_t) forward across periods (not just growth rates), since eq 11's
        # X_{t+1} needs the actual compounding wage-bill level.
        omega_dot_path = Vector{Matrix{Float64}}(undef, T+1)
        w_path = Vector{Matrix{Float64}}(undef, T+2)
        w_path[1] = w_0
        pi_current = pi_0
        for t in 0:T
            # Guard against 0/0: if a market's labor mass is (numerically) zero at t, its
            # L_dot doesn't matter -- the eq 12 constraint below multiplies it by L_t=0 anyway
            # -- but literal NaN would still poison that constraint's coefficient (0 * NaN =
            # NaN in IEEE floats, not 0). Use a finite placeholder (1.0) instead of dividing.
            L_dot_step = [
                L_path[t+1][n,j] == 0 ? 1.0 : L_path[t+2][n,j] / L_path[t+1][n,j]
                for n in 1:N, j in 1:M
            ]
            A_dot_step = (t+1 <= T) ? A_dot_path[t+1] : ones(N,J)
            kappa_dot_step = (t+1 <= T) ? kappa_dot_path[t+1] : [ones(N,N) for _ in 1:J]

            w_dot, P_dot, pi_next, X_next, status = solve_temp_eq_hat(
                pi_current, w_path[t+1], L_path[t+1], L_dot_step, A_dot_step, kappa_dot_step)

            P_hat_dot = [prod(P_dot[n,k]^alpha[k] for k in 1:J) for n in 1:N]
            omega_dot_path[t+1] = hcat(ones(N), w_dot ./ P_hat_dot)
            w_path[t+2] = w_dot .* w_path[t+1]
            pi_current = pi_next
        end
        pi_final = pi_current

        # Step 5: backward-solve u_dot_1,...,u_dot_T (eq 14), from the fixed terminal
        # u_dot_{T+1}=1 down to u_dot_1. u_dot_{t+1} needs mu_t (= mu_path[t+1]), omega_dot_{t+1}
        # (= omega_dot_path[t+1]), and u_dot_{t+2} (the previous, i.e. one-later, backward step).
        u_dot_new = Vector{Matrix{Float64}}(undef, T)
        u_dot_next = u_dot_terminal
        for t in (T-1):-1:0
            u_dot_new[t+1] = [
                omega_dot_path[t+1][n,j] * (sum(mu_path[t+1][n,j,i,k] * u_dot_next[i,k]^(beta/nu) for i in 1:N, k in 1:M))^nu
                for n in 1:N, j in 1:M
            ]
            u_dot_next = u_dot_new[t+1]
        end

        # Steps 6-7: compare and iterate (damped).
        diff = maximum(maximum(abs.(u_dot_new[t] .- u_dot_guess[t])) for t in 1:T)
        u_dot_guess = [damp .* u_dot_new[t] .+ (1-damp) .* u_dot_guess[t] for t in 1:T]
        converged = diff < tol
        outer_used = outer
        if converged
            println("solve_transition_path_hat converged after $outer outer iterations (max u_dot change = $diff)")
            break
        end
        if outer == max_outer
            println("solve_transition_path_hat: reached max_outer=$max_outer without converging (max u_dot change = $diff)")
        end
    end

    return (; L_path, mu_path, w_path, pi_final, u_dot_path = u_dot_guess, omega_dot_path, converged, outer_used)
end


solve_transition_path_hat (generic function with 1 method)

# Logging Dynamic Hat Solver Runs

Every call to `solve_transition_path_hat` below (routed through `run_shock_scenario`/`run_random_shock_scenario`/`run_simple_scenario` in the Scenario Infrastructure section) appends one row to `scenario_runs`: the conditions the solver was given -- the baseline allocations $(w_0,L_0,\pi_0,\mu_{-1})$, the fundamentals path $(\dot A_t,\dot\kappa_t)$, and the solver settings $(T,\text{max\_outer},\text{tol},\text{damp})$ -- alongside the full transition path it returned, so any scenario can be compared or inspected later without re-running the solver. A scenario run through one of the `with_*_override` helpers (Section D) also tags its row with `swept_param`/`swept_value` -- e.g. `(:tau_mig, 20.0)` -- so a parameter sweep can be pulled back out of the log with a `filter`, instead of matching exact label strings.

In [10]:
# migration_flows: from a scenario's mu_path/L_path (eqs 13/15), computes the gross and net
# migration flow into each market (n,j) at every period t=0,...,T. "Gross" is total churn through
# the market -- everyone who arrives from elsewhere PLUS everyone who leaves for elsewhere,
# excluding the (n,j)->(n,j) "stayers" term -- so it stays positive even once the economy has
# settled into a new steady state (idiosyncratic taste shocks keep people reshuffling), and is
# exactly zero only when mu_t^{nj,nj} = 1 for everyone (the no-mobility limit, tau_mig -> infty).
# "Net" is just L_{t+1}^{nj} - L_t^{nj} (inflow minus outflow), which -> 0 as the labor
# distribution converges to its new long-run allocation, regardless of how much churn remains.
function migration_flows(mu_path::Vector{Array{Float64,4}}, L_path::Vector{Matrix{Float64}})
    Tflow = length(mu_path) # one entry per t = 0,...,T, matching mu_path's indexing (path[s] = value at t=s-1)
    gross_path = Vector{Matrix{Float64}}(undef, Tflow)
    net_path = Vector{Matrix{Float64}}(undef, Tflow)
    for s in 1:Tflow
        mu_t, L_t, L_tp1 = mu_path[s], L_path[s], L_path[s+1]
        Nn, Mm = size(L_t)
        gross_in = [sum(mu_t[i,k,n,j]*L_t[i,k] for i in 1:Nn, k in 1:Mm if !(i==n && k==j)) for n in 1:Nn, j in 1:Mm]
        gross_out = [L_t[n,j]*(1 - mu_t[n,j,n,j]) for n in 1:Nn, j in 1:Mm]
        gross_path[s] = gross_in .+ gross_out
        net_path[s] = L_tp1 .- L_t
    end
    return gross_path, net_path
end

# scenario_runs: every scenario in this notebook -- every section below -- logs one row here via
# log_transition_run!. "Endogenous" refers to the model's own responses -- wages, labor, utility --
# as opposed to the exogenous fundamentals shock (A_dot/kappa_dot) driving them; see
# shock_end_period vs. endog_converged_period below for how the log distinguishes the two.
# swept_param/swept_value tag a row with which global parameter (if any) was varied to produce it
# -- e.g. (:tau_mig, 20.0) -- via one of the with_*_override helpers, run_taumig_variation_scenario,
# or an explicit tag at the call site; missing for scenarios that aren't part of a parameter sweep.
scenario_runs = DataFrame(
    label = String[],
    A_shocks = Any[],       # productivity shock specs (n,j,factor,ramp) passed to run_shock_scenario
    kappa_shocks = Any[],   # trade-cost shock specs (n,j,i,factor,ramp) passed to run_shock_scenario
    w_0 = Any[],             # baseline wage fed to the solver (N x J)
    L_0 = Any[],             # baseline labor distribution fed to the solver (N x M)
    pi_0 = Any[],            # baseline trade shares fed to the solver (N x J x N)
    mu_minus1 = Any[],      # mu_{-1} fed to the solver (N x M x N x M)
    A_dot_path = Any[],     # path of productivity changes, t = 1,...,T
    kappa_dot_path = Any[], # path of trade-cost changes, t = 1,...,T
    T = Int[],
    max_outer = Int[],
    tol = Float64[],
    damp = Float64[],
    converged = Bool[],
    outer_used = Int[],
    endog_tol = Float64[],                          # tolerance used to judge the endogenous path "converged"
    shock_end_period = Int[],                        # last period p in {0,...,T} where fundamentals still differ from 1
    endog_converged_period = Union{Missing,Int}[],    # first period p* s.t. w_dot/L_dot/u_dot stay within endog_tol
                                                       # of 1 for every period from p* through T; missing if it
                                                       # never happens within the T-period horizon
    endog_convergence_lag = Union{Missing,Int}[],     # endog_converged_period - shock_end_period; missing if the
                                                       # above is missing
    gross_migration_path = Any[],  # Vector of N x M matrices, one per period t=0,...,T: total churn
                                    # (in + out, excluding stayers) through each market (n,j) -- see migration_flows
    net_migration_path = Any[],    # Vector of N x M matrices, one per period t=0,...,T: L_{t+1}-L_t at each
                                    # market (n,j) -- should -> 0 as the labor distribution converges
    swept_param = Union{Missing,Symbol}[],  # which global parameter this run swept (e.g. :tau_mig), or missing
    swept_value = Any[],                    # the value it was set to for this run, or missing
    result = Any[],          # full named tuple from solve_transition_path_hat -- the estimated
                              # whole transition path (L_path, mu_path, w_path, pi_final,
                              # u_dot_path, omega_dot_path, converged, outer_used)
)

function log_transition_run!(df::DataFrame, label::String, A_shocks, kappa_shocks,
                              w_0, L_0, pi_0, mu_minus1, A_dot_path, kappa_dot_path,
                              T::Int, max_outer::Int, tol::Float64, damp::Float64, result;
                              endog_tol::Float64=1e-3,
                              swept_param::Union{Symbol,Missing}=missing, swept_value=missing)
    # Last period the fundamentals path itself was still changing (exact == 1.0 is safe: untouched
    # entries are literal `ones(...)`, never arithmetically touched, so no float-tolerance is needed).
    shock_end_period = 0
    for p in 1:length(A_dot_path)
        is_flat = A_dot_path[p] == ones(N,J) && all(kappa_dot_path[p][j] == ones(N,N) for j in 1:J)
        is_flat || (shock_end_period = p)
    end

    # Endogenous "dot" deviation from 1 at period p, using the same "change going into period p"
    # convention as A_dot_path/u_dot_path (w_path[p+1]/w_path[p] = w_p/w_{p-1}, since w_path[s]=w_{s-1}).
    Tu = length(result.u_dot_path)
    dev(p) = max(maximum(abs.(result.w_path[p+1] ./ result.w_path[p] .- 1)),
                 maximum(abs.(result.L_path[p+1] ./ result.L_path[p] .- 1)),
                 maximum(abs.(result.u_dot_path[p] .- 1)))

    # First period after which the endogenous path stays within tolerance through the end of the horizon.
    endog_converged_period = missing
    for p in 1:Tu
        if all(dev(q) < endog_tol for q in p:Tu)
            endog_converged_period = p
            break
        end
    end
    endog_convergence_lag = ismissing(endog_converged_period) ? missing : endog_converged_period - shock_end_period

    gross_migration_path, net_migration_path = migration_flows(result.mu_path, result.L_path)

    push!(df, (
        label=label, A_shocks=A_shocks, kappa_shocks=kappa_shocks,
        w_0=w_0, L_0=L_0, pi_0=pi_0, mu_minus1=mu_minus1,
        A_dot_path=A_dot_path, kappa_dot_path=kappa_dot_path,
        T=T, max_outer=max_outer, tol=tol, damp=damp,
        converged=result.converged, outer_used=result.outer_used,
        endog_tol=endog_tol, shock_end_period=shock_end_period,
        endog_converged_period=endog_converged_period, endog_convergence_lag=endog_convergence_lag,
        gross_migration_path=gross_migration_path, net_migration_path=net_migration_path,
        swept_param=swept_param, swept_value=swept_value,
        result=result,
    ))
    return gross_migration_path, net_migration_path
end

log_transition_run! (generic function with 1 method)

# Scenario Infrastructure

Shared helpers used by every scenario section below: building shock paths from a spec, generating i.i.d. shock paths, temporarily overriding a global parameter, and running/logging/summarizing a scenario. The scenario sections themselves (A-D) contain only the actual shocks being explored.

In [11]:
# print_labeled: formats an N x M(-ish) matrix with row/column labels for console display.
# Used below by summarize_transition to report labor distributions and wage changes.
function print_labeled(title, description, M::AbstractMatrix, colnames)
    println(title)
    println("  ", description)
    println("             " * join(rpad.(colnames, 16)))
    for n in 1:size(M,1)
        println("  Region $n:  " * join([rpad(string(round(M[n,c], digits=4)), 16) for c in 1:size(M,2)]))
    end
    println()
end


print_labeled (generic function with 1 method)

In [12]:
# build_shock_paths(A_shocks, kappa_shocks, Tsim): turns a list of productivity/trade-cost shock
# specs into the A_dot_path/kappa_dot_path arrays solve_transition_path_hat expects. Used by
# run_shock_scenario and run_simple_scenario.
#
# shock spec: (n=region, j=sector, factor=target multiplicative change, ramp=periods to phase in
# over -- 1 for a one-time jump). kappa shock spec adds i=origin region (kappa_dot[j][n,i], the
# cost shipping sector-j goods from i to n, matching the kappa_0 convention used throughout).
function build_shock_paths(A_shocks, kappa_shocks, Tsim::Int)
    A_dot_path = [ones(N,J) for _ in 1:Tsim] # starts at "no change" every period
    for s in A_shocks
        per_period = s.factor^(1/s.ramp) # constant per-period multiplier that compounds to s.factor over s.ramp periods
        for t in 1:min(s.ramp, Tsim)
            A_dot_path[t][s.n, s.j] *= per_period # applied only to periods 1..ramp; periods after
                                                    # ramp stay at 1 (no further change), so the
                                                    # shock's effect is permanent once fully phased in
        end
    end

    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim] # same "no change" starting point
    for s in kappa_shocks
        per_period = s.factor^(1/s.ramp)
        for t in 1:min(s.ramp, Tsim)
            kappa_dot_path[t][s.j][s.n, s.i] *= per_period
        end
    end

    return A_dot_path, kappa_dot_path
end

# iid_shock_paths(periods, sigma_A, sigma_kappa, Tsim; seed): every period in `periods` draws an
# independent log-normal multiplicative shock (mean 1 in log, so E[log(dot)]=0) for every
# region-sector productivity and every off-diagonal trade cost; periods outside that range are
# left at dot=1 (no change). `seed`, if given, reseeds the global RNG first so the draw is
# reproducible across re-runs. Used by run_random_shock_scenario, run_taumig_variation_scenario,
# and run_sector_scaling_scenario.
function iid_shock_paths(periods, sigma_A::Float64, sigma_kappa::Float64, Tsim::Int;
                          seed::Union{Int,Nothing}=nothing)
    seed === nothing || Random.seed!(seed)

    A_dot_path = [ones(N,J) for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        A_dot_path[p] = exp.(sigma_A .* randn(N,J)) # log-normal draw, one independent value per region-sector
    end

    kappa_dot_path = [[ones(N,N) for _ in 1:J] for _ in 1:Tsim]
    for p in periods
        p <= Tsim || continue
        for j in 1:J, n in 1:N, i in 1:N
            n == i && continue # self-shipping cost kappa[n,n] is never shocked
            kappa_dot_path[p][j][n,i] = exp(sigma_kappa * randn())
        end
    end

    shock_spec = (kind=:iid_random_walk, sigma_A=sigma_A, sigma_kappa=sigma_kappa, periods=periods, seed=seed)
    return A_dot_path, kappa_dot_path, shock_spec
end

iid_shock_paths (generic function with 1 method)

In [13]:
# recompute_mu_stationary(): rebuilds mu_stationary (the migration shares of the economy's own
# stationary equilibrium) from the CURRENT global tau_mig/beta/nu/b/A_0/kappa_0 and the baseline
# wage w_temp. Used to set the initial mu_stationary below, and after every with_*_override.
recompute_mu_stationary() = migration_shares(stationary_V(flow_utility_mkt_at(w_temp, A_0, kappa_0)))

# with_taumig_override(f, value): temporarily sets every OFF-DIAGONAL entry of tau_mig to `value`
# (the diagonal -- staying put -- is always 0), recomputes mu_stationary under it, runs f(), then
# restores tau_mig and mu_stationary -- even if f() throws.
function with_taumig_override(f::Function, value::Real)
    global tau_mig, mu_stationary
    tau_mig_original, mu_stationary_original = tau_mig, mu_stationary
    tau_mig = [(n == i && j == k) ? 0.0 : Float64(value) for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
    mu_stationary = recompute_mu_stationary()
    try
        return f()
    finally
        tau_mig, mu_stationary = tau_mig_original, mu_stationary_original
    end
end

# with_beta_override(f, value): temporarily sets the household discount factor beta (which feeds
# mu_stationary AND the transition dynamics directly -- see the household-block markdown above),
# recomputes mu_stationary under it, runs f(), then restores both.
function with_beta_override(f::Function, value::Float64)
    global beta, mu_stationary
    beta_original, mu_stationary_original = beta, mu_stationary
    beta = value
    mu_stationary = recompute_mu_stationary()
    try
        return f()
    finally
        beta, mu_stationary = beta_original, mu_stationary_original
    end
end

# with_nu_override(f, value): temporarily sets the migration taste-shock dispersion nu (which,
# like beta, feeds mu_stationary AND the transition dynamics directly), recomputes mu_stationary
# under it, runs f(), then restores both.
function with_nu_override(f::Function, value::Float64)
    global nu, mu_stationary
    nu_original, mu_stationary_original = nu, mu_stationary
    nu = value
    mu_stationary = recompute_mu_stationary()
    try
        return f()
    finally
        nu, mu_stationary = nu_original, mu_stationary_original
    end
end

# with_b_override(f, factor): temporarily scales the home-production value b by `factor`
# (e.g. factor=0.1 for a -90% cut), recomputes mu_stationary under it, runs f(), then restores both.
function with_b_override(f::Function, factor::Float64)
    global b, mu_stationary
    b_original, mu_stationary_original = b, mu_stationary
    b = b .* factor
    mu_stationary = recompute_mu_stationary()
    try
        return f()
    finally
        b, mu_stationary = b_original, mu_stationary_original
    end
end

with_b_override (generic function with 1 method)

A playground for running `solve_transition_path_hat` under different shocks, without needing to hand-build a `T`-length path of matrices every time. Every scenario below starts the economy at the same baseline -- `(w_temp, L_0, pi_temp)` from the Baseline Levels section above -- and treats its shock as an **unanticipated surprise hitting an economy that was previously at its own steady state**. Concretely, $\mu_{-1}$ is set to `mu_stationary`, the migration shares of the *baseline, unshocked* stationary equilibrium.

This is a simplification: true perfect-foresight $\mu_{-1}$ would need to be derived from the *shocked* economy's own period-0 value function, which requires already knowing the transition path -- not something a quick scenario exploration can do. Treating $\mu_{-1}$ as the baseline's own stationary migration shares is a reasonable stand-in for an unanticipated shock.

`run_shock_scenario` (below) takes lists of productivity shocks (`A_shocks`) and/or trade-cost shocks (`kappa_shocks`), each specifying a market, a target multiplicative change (`factor`), and how many periods it phases in over (`ramp` -- `1` for a one-time jump, more for a gradual change), solves the transition, and prints a before/after summary. `run_random_shock_scenario` is the i.i.d.-shock alternative used from Scenario 8 onward.

Every scenario cell in Sections A-D below has a markdown cell right above it stating what it **Tests** and, where it isn't obvious from the code, how it's **Implemented**; the code cell itself is commented on the mechanics of the call (what the parameters do), not the economics.

In [14]:
mu_stationary = recompute_mu_stationary() # baseline mu_{-1} for every scenario: the migration
    # shares of the economy's own stationary equilibrium under the unshocked baseline wage w_temp
    # (see the markdown above for why this is an approximation rather than exact foresight)

function summarize_transition(label::String, result, gross_migration_path, net_migration_path; Tsim::Int=T)
    println("\n", "="^70)
    println(label)
    println("="^70)
    print_labeled(
        "Labor distribution at t=0",
        "mass of households in each region-market, before the shock",
        result.L_path[1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Labor distribution at t=$Tsim (new long-run allocation)",
        "mass of households in each region-market, after the transition",
        result.L_path[Tsim+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Wage change w_$Tsim / w_temp",
        "nominal wage at t=$Tsim relative to the pre-shock baseline",
        result.w_path[Tsim+1] ./ w_temp, ["Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Net migration flow into t=$Tsim (L_$Tsim - L_$(Tsim-1))",
        "should -> 0 as the labor distribution converges",
        net_migration_path[Tsim], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
    print_labeled(
        "Gross migration flow into t=$Tsim",
        "total churn (in+out, excl. stayers) into each market -- stays bounded away from 0",
        gross_migration_path[Tsim], ["Non-employment","Sector 1","Sector 2","Sector 3"]
    )
end

# This is the function every scenario in Sections A-C calls, and where each scenario's
# A_shocks/kappa_shocks argument (see build_shock_paths above) turns into the transition path.
function run_shock_scenario(label::String; A_shocks=NamedTuple[], kappa_shocks=NamedTuple[], Tsim::Int=T,
                             max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5, endog_tol::Float64=1e-3,
                             swept_param::Union{Symbol,Missing}=missing, swept_value=missing)
    A_dot_path, kappa_dot_path = build_shock_paths(A_shocks, kappa_shocks, Tsim)

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    gross_migration_path, net_migration_path = log_transition_run!(scenario_runs, label, A_shocks, kappa_shocks,
                         w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp,
                         result; endog_tol=endog_tol, swept_param=swept_param, swept_value=swept_value)
    summarize_transition(label, result, gross_migration_path, net_migration_path; Tsim=Tsim)
    return result
end

# Alternative to run_shock_scenario for i.i.d. RANDOM (rather than deterministic ramped) shocks --
# see iid_shock_paths above for how the shock itself is drawn.
function run_random_shock_scenario(label::String; periods=1:50, sigma_A::Float64=0.05, sigma_kappa::Float64=0.05,
                                    Tsim::Int=T, seed::Union{Int,Nothing}=nothing,
                                    max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5, endog_tol::Float64=1e-3,
                                    swept_param::Union{Symbol,Missing}=missing, swept_value=missing)
    A_dot_path, kappa_dot_path, shock_spec = iid_shock_paths(periods, sigma_A, sigma_kappa, Tsim; seed=seed)

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    gross_migration_path, net_migration_path = log_transition_run!(scenario_runs, label, [shock_spec], [shock_spec],
                         w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp,
                         result; endog_tol=endog_tol, swept_param=swept_param, swept_value=swept_value)
    summarize_transition(label, result, gross_migration_path, net_migration_path; Tsim=Tsim)
    return result
end

run_random_shock_scenario (generic function with 1 method)

In [15]:
# Unlike summarize_transition (which only prints t=0 and t=Tsim), this reports every period in
# t_start:t_end: labor distribution, wage relative to the pre-shock baseline, and gross/net
# migration flow into each market. Used by run_simple_scenario (Section D's default reporting
# style).
function summarize_simple_scenario(label::String, result, gross_migration_path, net_migration_path;
                                    t_start::Int=1, t_end::Int=3)
    println("\n", "="^70)
    println(label)
    println("="^70)
    for t in t_start:t_end
        println("-- t=$t --")
        print_labeled(
            "Labor distribution",
            "mass of households in each region-market at t=$t",
            result.L_path[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
        )
        print_labeled(
            "Wage change w_$t / w_temp",
            "nominal wage at t=$t relative to the pre-shock baseline",
            result.w_path[t+1] ./ w_temp, ["Sector 1","Sector 2","Sector 3"]
        )
        print_labeled(
            "Net migration flow (L_$t - L_$(t-1))",
            "should -> 0 as the labor distribution converges",
            net_migration_path[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
        )
        print_labeled(
            "Gross migration flow",
            "total churn (in+out, excl. stayers) into each market -- stays bounded away from 0",
            gross_migration_path[t+1], ["Non-employment","Sector 1","Sector 2","Sector 3"]
        )
    end
end

# Same shock-spec convention as run_shock_scenario (via build_shock_paths), but with a Tsim=3
# default horizon and a period-by-period summary instead of summarize_transition's start/end
# snapshot -- Section D's default reporting style. swept_param/swept_value let a call tag itself
# as part of a parameter sweep (see the with_*_override helpers above) for later filtering.
function run_simple_scenario(label::String; A_shocks=NamedTuple[], kappa_shocks=NamedTuple[], Tsim::Int=3,
                              max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5, endog_tol::Float64=1e-3,
                              t_start::Int=1, t_end::Int=3,
                              swept_param::Union{Symbol,Missing}=missing, swept_value=missing)
    A_dot_path, kappa_dot_path = build_shock_paths(A_shocks, kappa_shocks, Tsim)

    result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    gross_migration_path, net_migration_path = log_transition_run!(scenario_runs, label, A_shocks, kappa_shocks,
                         w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp,
                         result; endog_tol=endog_tol, swept_param=swept_param, swept_value=swept_value)
    summarize_simple_scenario(label, result, gross_migration_path, net_migration_path; t_start=t_start, t_end=t_end)
    return result
end

run_simple_scenario (generic function with 1 method)

## A. Single- and Multi-Market Shocks

**Scenario 1 — No shock**

Tests how much labor reallocation happens with NO productivity or trade-cost change at all -- since $L_0$ isn't itself the stationary distribution implied by `mu_stationary`, there's still real convergence toward the baseline's own steady state. This is the reference point for judging how much of the change in the shocked scenarios below is "just settling down" versus caused by the shock.

In [16]:
# run_shock_scenario with no A_shocks/kappa_shocks: A_dot_path/kappa_dot_path stay at their
# default all-ones (no change) for every period.
scenario_none = run_shock_scenario("Scenario 1: No shock (pure convergence to baseline steady state)")

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 5.808641567739414e-9)

Scenario 1: No shock (pure convergence to baseline steady state)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0278          0.1555          0.3684          0.4483          
  Region 2:  3.0278          0.4862          0.3403          0.1458          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.686191223606516 0.22270326478503655 0.49616374488216036 0.5958656020149583; 2.686191223606516 0.6428735233636291 0.4612885794337561 0.20872283830742924], [2.9662416664911198 0.16377223697566004 0.3920212818295205 0.47785385626323823; 2.9662416664911198 0.5186223639862972 0.36176011214668985 0.15348681581635615], [3.016467509061154 0.15684066376086198 0.3727672887989706 0.4538461440829731; 3.016467509061154 0.49238644950793337 0.3441670572788329 0.1470573784481225], [3.0256702471806425 0.1557510860772527 0.36922986715896416 0.44929978922970304; 3.0256702471806425 0.48735164103539175 0.3409947112957242 0.1460324108416822], [3.027366261735465 0.15555938303913208 0.36857808945810144 0.44845444398585477; 3.027366261735465 0.48641065813058904 0.34041455292606465 0.14585034898932936], [3.0276793289128605 0.1555244700205106 0.36845779665649997 0.44829802767081606; 3.0276793289128605 0.4862362015156075 0.3403077910024702 0.1458170553083775], [3.

**Scenario 2 — Region 1, Sector 1 productivity +20% (one-time jump)**

Tests how a one-sided productivity gain in a single market reallocates labor and wages.

In [17]:
scenario_up = run_shock_scenario("Scenario 2: +20% productivity, region 1 sector 1 (one-time jump)";
    A_shocks = [(n=1, j=1, factor=1.2, ramp=1)])
    # n=region, j=sector, factor=target multiplicative change on A_0[n,j] (1.2 = +20%),
    # ramp=periods to phase the change in over (1 = applied entirely in period 1)

solve_transition_path_hat converged after 20 outer iterations (max u_dot change = 8.12633360602888e-9)

Scenario 2: +20% productivity, region 1 sector 1 (one-time jump)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0174          0.174           0.3703          0.4506          
  Region 2:  3.0174          0.4817          0.342           0.1465          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.6749477818462064 0.24719418900980827 0.497584811703744 0.5975642719445258; 2.6749477818462064 0.6358159149114195 0.4626125957446996 0.2093326529933899], [2.9556253370850585 0.18340715441973254 0.39389001596268647 0.48012941705730205; 2.9556253370850585 0.5136184116661995 0.36348519240046223 0.1542191343234998], [3.006072168012991 0.17551131922260238 0.374658333098933 0.4561551834873423; 3.006072168012991 0.48783053151787004 0.345910045111709 0.14779025153556027], [3.0153231584732243 0.17425632728625545 0.3711184950263351 0.4516060755444784; 3.0153231584732243 0.48287312735147414 0.3427353473396824 0.14676431050532482], [3.0170292207937424 0.17403533676618274 0.37046557177405776 0.45075921800599117; 3.0170292207937424 0.48194526265124393 0.34215417706177514 0.1465819921532634], [3.017344359520465 0.17399509326149415 0.3703449697182774 0.4506023858066896; 3.017344359520465 0.4817730508142469 0.34204714692869453 0.14654863442966637], [3.01

**Scenario 3 — Region 1, Sector 1 productivity -20% (one-time decline)**

The mirror image of Scenario 2 -- a one-sided competitiveness loss in a single market (in the spirit of a "China shock"-style negative shock), to see whether the response is symmetric.

In [18]:
scenario_down = run_shock_scenario("Scenario 3: -20% productivity, region 1 sector 1 (one-time decline)";
    A_shocks = [(n=1, j=1, factor=0.8, ramp=1)])
    # same fields as Scenario 2's A_shocks; factor=0.8 (< 1) makes this a cut instead of a gain

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 9.840530612592602e-9)

Scenario 3: -20% productivity, region 1 sector 1 (one-time decline)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0387          0.1353          0.3667          0.4462          
  Region 2:  3.0387          0.4906          0.3387          0.1451          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Regi

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.698253459336259 0.19550816002359253 0.49500803502782587 0.5944850375380659; 2.698253459336259 0.6500552258452947 0.4602114125232383 0.2082252103694653], [2.977462595873722 0.14234856848087613 0.3903274655091282 0.47579475222220613; 2.977462595873722 0.5235917627086548 0.3601951305446792 0.15281712878701087], [3.027450419184213 0.13645913339956442 0.3710397331058076 0.45173876351299425; 3.027450419184213 0.49690315362900755 0.3425739090855879 0.14638446889861156], [3.03660525763501 0.1355420589696896 0.36750225217999444 0.4471918410431931; 3.03660525763501 0.49179160271365224 0.339401750201937 0.1453599796215126], [3.038291768178292 0.13538057387911379 0.3668510133613736 0.4463471982779784; 3.038291768178292 0.49083756561008546 0.33882207419717636 0.14517803831768705], [3.0386029417008324 0.13535114104085932 0.3667308885914738 0.44619101018520796; 3.0386029417008324 0.49066084424219253 0.33871545713421813 0.14514477540438264], [3.0386603

**Scenario 4 — Region 1, Sector 1 productivity +20%, ramped over 5 periods**

The same long-run shock as Scenario 2, but phased in gradually -- compares transition speed/shape against Scenario 2's one-time jump (both should reach close to the same long-run allocation, just arriving at a different pace).

In [19]:
scenario_ramp = run_shock_scenario("Scenario 4: +20% productivity, region 1 sector 1 (ramped over 5 periods)";
    A_shocks = [(n=1, j=1, factor=1.2, ramp=5)])
    # ramp=5: run_shock_scenario spreads the 20% target across 5 periods as a constant per-period
    # multiplier (1.2^(1/5)) rather than applying it all in period 1

solve_transition_path_hat converged after 21 outer iterations (max u_dot change = 6.080586256729248e-9)

Scenario 4: +20% productivity, region 1 sector 1 (ramped over 5 periods)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  3.0161          0.1748          0.3708          0.4512          
  Region 2:  3.0161          0.4819          0.3425          0.1467          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.6837134725907803 0.22766959680525076 0.49652554650056396 0.5963028599039891; 2.6837134725907803 0.6415810938262441 0.46162376953425915 0.20887018824813397], [2.9618264282147915 0.171535712672075 0.3928364126646868 0.47885327583236853; 2.9618264282147915 0.516815888769519 0.3625098632793404 0.15379599035242741], [3.00979000813395 0.16817157634939672 0.37406237186710883 0.4554355092247824; 3.00979000813395 0.4898456321471193 0.34535758929235866 0.14754730485133408], [3.016603087248206 0.17096236160274864 0.3710245621898469 0.45150229784519413; 3.016603087248206 0.483948685272449 0.34264452742161616 0.14671139117173443], [3.016080077289137 0.17461433261483608 0.3708072768943768 0.4511888827656715; 3.016080077289137 0.48206928227281576 0.3424642971530589 0.14669577372096837], [3.0160657323159508 0.17478290520960033 0.3708038691136056 0.4511788711302012; 3.0160657323159508 0.48193695632225797 0.34246381614861476 0.14670211744382086], [3.0160

**Scenario 5 — Sector 1 productivity +20% in BOTH regions (symmetric)**

Tests whether a shock that leaves relative productivity unchanged (both regions gain equally) leaves trade shares and migration incentives roughly undisturbed, unlike Scenario 2's one-sided version -- migration and trade only respond to relative, not absolute, productivity.

In [20]:
scenario_symmetric = run_shock_scenario("Scenario 5: +20% productivity in sector 1, BOTH regions (symmetric)";
    A_shocks = [(n=1, j=1, factor=1.2, ramp=1), (n=2, j=1, factor=1.2, ramp=1)])
    # two shock specs, one per region -- each a one-time +20% jump to that region's A_0[.,1]

solve_transition_path_hat converged after 24 outer iterations (max u_dot change = 7.960827330677489e-9)

Scenario 5: +20% productivity in sector 1, BOTH regions (symmetric)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.9744          0.1637          0.3887          0.4731          
  Region 2:  2.9744          0.5132          0.3589          0.1535          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Regi

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.634138522861165 0.23161376018480395 0.5158166007634007 0.6194031722187383; 2.634138522861165 0.6682357069360647 0.47958023796194693 0.21707347621271456], [2.911687815086448 0.17219891552996822 0.4127548083802765 0.5032371885151034; 2.911687815086448 0.5462086161298534 0.3808456048120472 0.1613792364598541], [2.962647628385161 0.16512631839746733 0.39321804411318284 0.478912015147753; 2.962647628385161 0.5196456093809051 0.3629774018686724 0.1548253543216944], [2.9722127371677938 0.16399664419535595 0.3895401370578268 0.4741845221234615; 2.9722127371677938 0.5144109263102868 0.35967893045288424 0.15376336552459502], [2.974019021454343 0.1637936734473093 0.3888457451202314 0.4732832584501105; 2.974019021454343 0.5134075513890166 0.3590611067880876 0.15357062189655654], [2.9743607002510006 0.1637558383983087 0.3887144140630717 0.47311233131149283; 2.9743607002510006 0.513216856930506 0.35894462507715086 0.15353453371746562], [2.97442536396

**Scenario 6 — Trade liberalization: iceberg cost region 2 → region 1, sector 1, -30%**

A pure trade-cost shock with no productivity change, exercising the `kappa_shocks` side of `run_shock_scenario` (every scenario above only used `A_shocks`).

In [21]:
scenario_trade = run_shock_scenario("Scenario 6: 30% reduction in trade costs, region 2 -> region 1, sector 1";
    kappa_shocks = [(n=1, j=1, i=2, factor=0.7, ramp=1)])
    # n=destination region, j=sector, i=origin region, factor/ramp as in A_shocks -- this spec
    # targets kappa_0[j][n,i], the cost shipping sector-j goods FROM i TO n

solve_transition_path_hat converged after 24 outer iterations (max u_dot change = 6.216107406586957e-9)

Scenario 6: 30% reduction in trade costs, region 2 -> region 1, sector 1
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=10 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.9822          0.1607          0.4057          0.4869          
  Region 2:  2.9822          0.4988          0.3399          0.1437          

Wage change w_10 / w_temp
  nominal wage at t=10 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.642436165015354 0.22756541278679 0.5370302147113458 0.637175592419738; 2.642436165015354 0.6520590257076302 0.4569884485528831 0.20430897579090523], [2.9196009296186376 0.1690209121886385 0.43111112323759854 0.5182423240281789; 2.9196009296186376 0.5309612319363908 0.36052018585788137 0.1509423635140368], [2.970426367905451 0.16207296687541445 0.41051134563500036 0.49299934765004; 2.970426367905451 0.5050627063202673 0.34363281776344695 0.1448680799449289], [2.9799618544885442 0.16095752777578884 0.40659143021482635 0.4880733282984114; 2.9799618544885442 0.4999977531709189 0.34056216302939873 0.14389408853356653], [2.981761788242732 0.16075636261669737 0.40584817755393077 0.48713310260612863; 2.981761788242732 0.4990302260643416 0.3399907050440985 0.14371784962933798], [2.9821021122768174 0.16071880719397832 0.40570739608147155 0.48695476207697497; 2.9821021122768174 0.4988466481894958 0.3398832694881664 0.1436848924162782], [2.98216648

## B. Economy-Wide Shocks

**Scenario 7 — "Everything" shock: every region-sector productivity +20% and every off-diagonal trade cost -30%, ramped over 50 periods**

The full-economy version of the shocks above, run out far enough ($T_{sim}=200$) to see how long the endogenous path (wages, labor, utility) takes to settle once the fundamentals stop moving at period 50.

In [22]:
# one shock spec per region-sector (productivity) -- every (n,j) pair
A_shocks_everything = vec([(n=n, j=j, factor=1.2, ramp=50) for n in 1:N, j in 1:J])
# one shock spec per region-sector-origin triple, excluding the self-shipping case n==i
kappa_shocks_everything = [(n=n, j=j, i=i, factor=0.7, ramp=50) for n in 1:N, j in 1:J, i in 1:N if n != i]

scenario_everything = run_shock_scenario("Scenario 7: all fundamentals shocked, ramped over 50 periods";
    A_shocks = A_shocks_everything, kappa_shocks = kappa_shocks_everything, Tsim = 200)
    # Tsim=200: simulation horizon in periods (the shock itself finishes ramping at period 50)


solve_transition_path_hat converged after 26 outer iterations (max u_dot change = 9.24135745705712e-9)

Scenario 7: all fundamentals shocked, ramped over 50 periods
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.5579          0.2475          0.5401          0.655           
  Region 2:  2.5579          0.7102          0.5012          0.2301          

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.679033433800976 0.2239009663063003 0.49889850838381156 0.5991563786245071; 2.679033433800976 0.6463983477672601 0.46376926576532274 0.20980966555084463], [2.9525661630631115 0.16589219379803818 0.3972581948370695 0.4842648164092123; 2.9525661630631115 0.5255465849198239 0.3664871027745842 0.15541878113504828], [2.995835690105074 0.16005067264935124 0.3806565246404253 0.4635085538936447; 2.995835690105074 0.5028305429392881 0.351294280448988 0.14998804521815473], [2.997712608589907 0.16012137360575227 0.3799092674992391 0.46237822740399886; 2.997712608589907 0.5014916016371316 0.3506499958635761 0.1500243168104897], [2.9918776541981233 0.16113033946286387 0.3821214820653941 0.4650397974600743; 2.9918776541981233 0.5043459529864418 0.3526667508013076 0.15094036882767267], [2.984533548502104 0.1623278330932644 0.3849080236472212 0.4684413162638561; 2.984533548502104 0.5080226625155971 0.35519867257539517 0.1520343949004605], [2.97683525654

**Scenario 8 — Same scope as Scenario 7, but i.i.d. random shocks instead of a smooth ramp**

Tests whether random period-to-period fundamentals (as opposed to Scenario 7's deterministic ramp) change how long the endogenous path takes to settle once the fundamentals stop moving.

In [23]:
scenario_random = run_random_shock_scenario("Scenario 8: all fundamentals, i.i.d. random shocks over periods 1-50";
    periods = 1:50, sigma_A = 0.05, sigma_kappa = 0.05, Tsim = 200, seed = 2)
    # periods=1:50: draw an independent shock every period in this range (fundamentals held flat,
    # dot=1, outside it); sigma_A/sigma_kappa=0.05: log-normal spread of the productivity/trade-cost
    # draws; Tsim=200: simulation horizon; seed=2: fixes the random draw so later scenarios can
    # reproduce this exact realization


solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 8.434787757138906e-9)

Scenario 8: all fundamentals, i.i.d. random shocks over periods 1-50
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6102          0.2003          0.5429          0.6257          
  Region 2:  2.6102          0.7627          0.4702          0.1778          

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.682657717418846 0.21062674199206097 0.4930479714621486 0.590672714115908; 2.682657717418846 0.6504691435250411 0.46690450683444884 0.222963487232699], [2.98216967723223 0.146644175454939 0.39810900788799336 0.4633809721543658; 2.98216967723223 0.5176124107256724 0.3429338964365744 0.16698018287599234], [3.0067663880718833 0.1506254799628298 0.3892355392705267 0.44711208881000325; 3.0067663880718833 0.5034567583725185 0.33099900607976424 0.16503835136059042], [3.019475245316762 0.13442100597245696 0.39010109912204277 0.44170428670667816; 3.019475245316762 0.5018752877670238 0.3231241462320975 0.16982368356617766], [3.005761479983189 0.13162081309086981 0.4004673519167107 0.44762658140340383; 3.005761479983189 0.5120375845243724 0.32293939063417293 0.17378531846409273], [2.987167218700723 0.1308672491441078 0.4155977779574943 0.45701573434786535; 2.987167218700723 0.5245995491520029 0.3210496369985775 0.17653561499850678], [2.954311601998

## C. Mobility Frictions & Economy Size

Scenarios 9-11 below probe how the fundamentals-to-endogenous convergence lag (see the Logging
section) responds to (a) mobility frictions and (b) economy size, all using the **same shock
process as Scenario 8**: i.i.d. log-normal productivity/trade-cost shocks, `sigma_A = sigma_kappa
= 0.05`, active over periods 1-50, `seed = 2`, simulated out to `Tsim = 200` (the same 150-period
post-shock buffer).

**Larger economies (e.g. N=10 regions) are not covered here**: the feasibility-based formulation
used by `solve_temporary_equilibrium`/`solve_temp_eq_hat` becomes numerically unreliable at that
scale.

Scenario 9 keeps `N=2, J=3` (Scenarios 1-8's baseline `w_temp`/`pi_temp`/`L_0`/`A_0`/`kappa_0`
untouched) and only overrides the *off-diagonal* mobility cost `tau_mig` (via `with_taumig_override`
from the Scenario Infrastructure section) -- the cost of staying in the same market remains 0, per
the paper's convention. Scenario 10 mirrors it with a near-infinite mobility cost. Scenario 11
regenerates the fundamentals at `J=15` (`N` unchanged at 2); because that requires reassigning the
shared globals (`N`, `J`, `M`, `A_0`, `kappa_0`, ...), it's run **last** and permanently leaves the
notebook's state at `N=2, J=15` -- re-run the Parameters/Baseline Levels cells above if you want to
get back to the `J=3` baseline afterward.

Section D revisits mobility-cost sensitivity again, at a much smaller shock and with
period-by-period reporting -- see its intro for how the two relate.

In [24]:
# run_taumig_variation_scenario: reruns the transition under a modified off-diagonal mobility
# cost (0 = free mobility, a large value = effectively no mobility), holding the production side
# (w_temp, pi_temp, L_0, A_0, kappa_0) fixed at the Scenario 1-8 baseline -- built on
# with_taumig_override (Scenario Infrastructure) and iid_shock_paths, reproducing the exact same
# shock realization as Scenario 8 for a given seed so tau_mig is the only thing that differs.
function run_taumig_variation_scenario(label::String, cost_value::Float64;
                                        periods=1:50, sigma_A::Float64=0.05, sigma_kappa::Float64=0.05,
                                        Tsim::Int=T, seed::Int=2, max_outer::Int=100, tol::Float64=1e-8,
                                        damp::Float64=0.5, endog_tol::Float64=1e-3)
    with_taumig_override(cost_value) do
        A_dot_path, kappa_dot_path, shock_spec = iid_shock_paths(periods, sigma_A, sigma_kappa, Tsim; seed=seed)

        result = solve_transition_path_hat(w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path;
                                            T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
        gross_migration_path, net_migration_path = log_transition_run!(scenario_runs, label, [shock_spec], [shock_spec],
                             w_temp, L_0, pi_temp, mu_stationary, A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp,
                             result; endog_tol=endog_tol, swept_param=:tau_mig, swept_value=cost_value)
        summarize_transition(label, result, gross_migration_path, net_migration_path; Tsim=Tsim)
        return result
    end
end

run_taumig_variation_scenario (generic function with 1 method)

**Scenario 9 — Migration costs = 0 everywhere off-diagonal (frictionless mobility)**

Tests how mobility frictions affect the convergence lag, holding the shock fixed -- same fundamentals-shock realization as Scenario 8 (same N, J, seed=2), so any difference from Scenario 8's convergence lag is attributable to mobility frictions alone.

In [25]:
scenario_taumig_zero = run_taumig_variation_scenario(
    "Scenario 9: migration costs = 0 everywhere (frictionless mobility)", 0.0; Tsim=200)
    # 0.0: the off-diagonal tau_mig value run_taumig_variation_scenario overrides to (diagonal,
    # i.e. staying put, is always 0 regardless); Tsim=200: simulation horizon, matching Scenario 8


solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 5.557526883137598e-9)

Scenario 9: migration costs = 0 everywhere (frictionless mobility)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.0911          0.3169          0.7396          0.8349          
  Region 2:  2.0911          0.9865          0.6582          0.2818          

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Re

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.41454978015264 0.26468442363639694 0.5911425975251449 0.7020118224489921; 2.41454978015264 0.767370650827458 0.5663163791159148 0.2793745661408131], [2.4465806949231057 0.25114813432700694 0.6037119402117792 0.685425267481895; 2.4465806949231057 0.75486867326646 0.5288434421940281 0.28284115267261967], [2.406342300748488 0.27000953698907004 0.6182559735013186 0.6968394421930754; 2.406342300748488 0.7724271065019059 0.5390386112740606 0.2907447280435945], [2.4215638757028333 0.2417791830348236 0.6211895207963437 0.690989236113272; 2.4215638757028333 0.7739190916769316 0.5291491456945872 0.299846071278378], [2.4047483751133667 0.23719993128268593 0.6345081642725264 0.698182448203837; 2.4047483751133667 0.7866297566191994 0.5286035044832499 0.3053794449117697], [2.390908206487598 0.23434724223025438 0.6513395358547122 0.7058136805498294; 2.390908206487598 0.7975143991122431 0.5216182162417501 0.30755051303601727], [2.3654333016667892 0.240

**Scenario 10 — Migration costs = 100000 everywhere off-diagonal (effectively no mobility)**

The opposite extreme from Scenario 9 -- households are pinned to their initial market since moving is prohibitively costly. Same shock realization as Scenario 8 and Scenario 9.

In [26]:
scenario_taumig_high = run_taumig_variation_scenario(
    "Scenario 10: migration costs = 100000 everywhere (effectively no mobility)", 100000.0; Tsim=200)
    # 100000.0: the off-diagonal tau_mig value; Tsim=200 as in Scenario 9


solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 6.2722032012629825e-9)

Scenario 10: migration costs = 100000 everywhere (effectively no mobility)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Labor distribution at t=200 (new long-run allocation)
  mass of households in each region-market, after the transition
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Wage change w_200 / w_temp
  nominal wage at t=200 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3    

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]  …  [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]], mu_path = [[1.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 1.0 0.0 0.0 0.0;;;; 0.0 1.0 0.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0;;;; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 0.0 0.0 1.0 0.0;;;; 0.0 0.0 0.0 1.0; 0.0 0.0 0.0 0.0;;

In [27]:
# run_sector_scaling_scenario: rebuilds the fundamentals at a new sector count J_new (N unchanged),
# re-solves the temporary-equilibrium baseline and mu_stationary from scratch (mirroring the
# Parameters and Baseline Levels cells above), then runs the same i.i.d. shock process as
# Scenario 8 (via iid_shock_paths). Unlike with_taumig_override, this permanently reassigns the
# shared globals (N, J, M, A_0, kappa_0, ...) since the production side itself is changing size --
# that's why it's run last, after the tau_mig scenarios have already used the original J=3 baseline.
#
# Uses its own summary logic below rather than the shared `summarize_transition` helper, since
# that helper assumes J=3 sector labels and reads the global w_temp (the J=3 baseline).
function run_sector_scaling_scenario(label::String, J_new::Int;
                                      seed_fundamentals::Int=1, periods=1:50, sigma_A::Float64=0.05,
                                      sigma_kappa::Float64=0.05, Tsim::Int=T, seed_shock::Int=2,
                                      max_outer::Int=100, tol::Float64=1e-8, damp::Float64=0.5,
                                      endog_tol::Float64=1e-3)
    global N, J, M, L_0, A_0, B, w_0, b, kappa_0, theta, eta, gamma, tau_mig, alpha
    J = J_new
    M = J + 1

    Random.seed!(seed_fundamentals)
    L_0 = ones(N, M)
    A_0 = rand(N, J)
    B = ones(N, J)
    w_0 = ones(N, J)
    b = ones(N)
    kappa_0 = [ones(N,N) for _ in 1:J]
    theta = fill(4.0, J)
    eta = fill(2.0, N, J)
    gamma = ones(N, J)
    tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]
    alpha = rand(J); alpha = alpha ./ sum(alpha)

    w_temp_v, pi_temp_v, X_temp_v, status_temp_v = solve_temporary_equilibrium(L_0)
    println("  [J=$J_new] baseline temporary-equilibrium solver status: ", status_temp_v)

    U_mkt_v = flow_utility_mkt_at(w_temp_v, A_0, kappa_0)
    V_v = stationary_V(U_mkt_v)
    mu_v = migration_shares(V_v)

    A_dot_path, kappa_dot_path, shock_spec = iid_shock_paths(periods, sigma_A, sigma_kappa, Tsim; seed=seed_shock)

    result = solve_transition_path_hat(w_temp_v, L_0, pi_temp_v, mu_v, A_dot_path, kappa_dot_path;
                                        T=Tsim, max_outer=max_outer, tol=tol, damp=damp)
    gross_migration_path, net_migration_path = log_transition_run!(scenario_runs, label, [shock_spec], [shock_spec],
                         w_temp_v, L_0, pi_temp_v, mu_v, A_dot_path, kappa_dot_path, Tsim, max_outer, tol, damp,
                         result; endog_tol=endog_tol)

    println("\n", "="^70)
    println(label)
    println("="^70)
    labor_colnames = vcat(["Non-employment"], ["Sector $j" for j in 1:J])
    print_labeled("Labor distribution at t=0", "mass of households in each region-market, before the shock",
                  result.L_path[1], labor_colnames)
    print_labeled("Labor distribution at t=$Tsim (new long-run allocation)",
                  "mass of households in each region-market, after the transition",
                  result.L_path[Tsim+1], labor_colnames)
    wage_colnames = ["Sector $j" for j in 1:J]
    print_labeled("Wage change w_$Tsim / w_temp", "nominal wage at t=$Tsim relative to the pre-shock baseline",
                  result.w_path[Tsim+1] ./ w_temp_v, wage_colnames)
    print_labeled("Net migration flow into t=$Tsim (L_$Tsim - L_$(Tsim-1))",
                  "should -> 0 as the labor distribution converges",
                  net_migration_path[Tsim], labor_colnames)
    print_labeled("Gross migration flow into t=$Tsim",
                  "total churn (in+out, excl. stayers) into each market -- stays bounded away from 0",
                  gross_migration_path[Tsim], labor_colnames)

    return result
end

run_sector_scaling_scenario (generic function with 1 method)

**Scenario 11 — J=15 sectors (N=2 unchanged)**

Tests how economy size (many more sectors to trade and migrate across) affects the convergence lag, using the same i.i.d. shock process as Scenario 8 (same statistical parameters and seed=2, but necessarily a different draw since `randn()` is now called at a different array size).

Regenerates `A_0`, `kappa_0`, etc. at the new size and re-solves the baseline from scratch; this **permanently reassigns** the shared globals (`N`, `J`, `M`, `A_0`, `kappa_0`, ...) to the 15-sector economy -- re-run the Parameters/Baseline Levels cells above to return to the `J=3` baseline.

In [28]:
scenario_sectors15 = run_sector_scaling_scenario(
    "Scenario 11: J=15 sectors (N=2)", 15; Tsim=200)
    # 15: the new J (sector count) to rebuild the fundamentals at; Tsim=200 as in Scenario 8


  [J=15] baseline temporary-equilibrium solver status: LOCALLY_SOLVED
solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 6.803137608812904e-9)

Scenario 11: J=15 sectors (N=2)
Labor distribution at t=0
  mass of households in each region-market, before the shock
             Non-employment  Sector 1        Sector 2        Sector 3        Sector 4        Sector 5        Sector 6        Sector 7        Sector 8        Sector 9        Sector 10       Sector 11       Sector 12       Sector 13       Sector 14       Sector 15       
  Region 1:  1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0             1.0      

(L_path = [[1.0 1.0 … 1.0 1.0; 1.0 1.0 … 1.0 1.0], [10.305755768941703 0.17296337283030333 … 0.404076924775912 0.12048902718542369; 10.305755768941703 0.3160559321074949 … 0.6671555228752206 0.2002029800762689], [11.504738908900036 0.12484108985867737 … 0.3182531448379036 0.08679921775543321; 11.504738908900036 0.253934420942132 … 0.5311241608126288 0.1608844676080147], [11.65098871067111 0.1316105990310174 … 0.32152539889465315 0.08710482732779051; 11.65098871067111 0.23973807871747477 … 0.5034056520053666 0.15387055752903672], [11.65640666122915 0.13427444884011638 … 0.32120163258896933 0.08868499350143998; 11.65640666122915 0.23789795890028917 … 0.5025953922414719 0.15275600303614467], [11.62483012663942 0.13483898703195768 … 0.3221826477548319 0.0899802551070532; 11.62483012663942 0.23968270829961752 … 0.5067209301436335 0.15337791858287736], [11.634471115467925 0.12738863573556825 … 0.33266969513032146 0.0880018116894115; 11.634471115467925 0.2424433768057719 … 0.4970557862485815 

In [29]:
# Convergence-lag summary for Scenarios 8-11 (same shock process throughout; only tau_mig or J
# differs), pulled straight from the scenario_runs log.
summary_cols = [:label, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
println(scenario_runs[in(["Scenario 8: all fundamentals, i.i.d. random shocks over periods 1-50",
                             "Scenario 9: migration costs = 0 everywhere (frictionless mobility)",
                             "Scenario 10: migration costs = 100000 everywhere (effectively no mobility)",
                             "Scenario 11: J=15 sectors (N=2)"]).(scenario_runs.label), summary_cols])


4×4 DataFrame
 Row │ label                              shock_end_period  endog_converged_period  endog_convergence_lag 
     │ String                             Int64             Union{Missing, Int64}   Union{Missing, Int64} 
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Scenario 8: all fundamentals, i.…                50                      52                      2
   2 │ Scenario 9: migration costs = 0 …                50                      51                      1
   3 │ Scenario 10: migration costs = 1…                50                      51                      1
   4 │ Scenario 11: J=15 sectors (N=2)                  50                      52                      2


In [30]:
# Migration flow diagnostics: for every scenario, show the max-abs NET flow (should shrink toward
# 0 as the labor distribution settles) and max GROSS flow (should stay bounded away from 0 for any
# finite tau_mig -- idiosyncratic taste shocks keep people reshuffling even at the new steady
# state -- and should be ~0 throughout only in the no-mobility scenario) at the start, middle, and
# final period of each scenario's horizon.
println("\n==== MIGRATION FLOW DIAGNOSTICS (max over all markets n,j) ====")
for row in eachrow(scenario_runs)
    gp, np = row.gross_migration_path, row.net_migration_path
    Tlen = length(gp)
    mid = max(1, Tlen ÷ 2)
    println(row.label)
    println("  net  max|.|  -- t=1: ", round(maximum(abs.(np[1])), digits=6),
            "   t=$mid: ", round(maximum(abs.(np[mid])), digits=6),
            "   t=$Tlen (final): ", round(maximum(abs.(np[end])), digits=8))
    println("  gross max    -- t=1: ", round(maximum(gp[1]), digits=6),
            "   t=$mid: ", round(maximum(gp[mid]), digits=6),
            "   t=$Tlen (final): ", round(maximum(gp[end]), digits=8))
    println()
end



==== MIGRATION FLOW DIAGNOSTICS (max over all markets n,j) ====
Scenario 1: No shock (pure convergence to baseline steady state)
  net  max|.|  -- t=1: 1.686191   t=5: 0.001696   t=11 (final): 7.0e-8
  gross max    -- t=1: 2.514671   t=5: 2.509507   t=11 (final): 2.50953249

Scenario 2: +20% productivity, region 1 sector 1 (one-time jump)
  net  max|.|  -- t=1: 1.674948   t=5: 0.001706   t=11 (final): 7.0e-8
  gross max    -- t=1: 2.506435   t=5: 2.509934   t=11 (final): 2.50996636

Scenario 3: -20% productivity, region 1 sector 1 (one-time decline)
  net  max|.|  -- t=1: 1.698253   t=5: 0.001687   t=11 (final): 7.0e-8
  gross max    -- t=1: 2.523526   t=5: 2.508898   t=11 (final): 2.50891689

Scenario 4: +20% productivity, region 1 sector 1 (ramped over 5 periods)
  net  max|.|  -- t=1: 1.683713   t=5: 0.003652   t=11 (final): 0.0
  gross max    -- t=1: 2.512856   t=5: 2.50989   t=11 (final): 2.50995726

Scenario 5: +20% productivity in sector 1, BOTH regions (symmetric)
  net  max|.

## D. Fine-Grained Parameter Sweeps

Revisits mobility-cost sensitivity from Section C (Scenarios 9-10), but at a much smaller shock
(a single-market +50% productivity shock, vs. Section C's economy-wide i.i.d. shocks) and with
period-by-period reporting instead of only a start/end snapshot -- which is why it reruns its own
Control/Low/High rather than reusing Section C's results directly. Also sweeps `beta` and `nu`,
which Section C doesn't touch at all.

Hand-picked scenarios explored one at a time (as opposed to the batch sweeps in Sections A-C),
with a period-by-period summary for the first few periods after each shock. As above, each
scenario cell states what it **Tests** and how that's **Implemented**.

In [31]:
# First cell of Section D: undoes Scenario 11's permanent reassignment of the shared globals
# (N, J, M, A_0, kappa_0, tau_mig, ...) back to the original N=2, J=3 baseline (same seed=1 draw
# order as the Parameters cell), and re-solves w_temp/pi_temp/mu_stationary from scratch -- every
# time this cell runs, regardless of what ran above it.
global N, J, M, L_0, A_0, B, w_0, b, kappa_0, theta, eta, gamma, beta, tau_mig, alpha, nu, T

N = 2
J = 3
M = J + 1

L_0 = ones(N,M)

Random.seed!(1)
A_0 = rand(N,J)

B = ones(N,J)
w_0 = ones(N,J)
b = ones(N)
kappa_0 = [ones(N,N) for _ in 1:J]

theta = fill(4.0, J)
eta = fill(2.0, N, J)
gamma = ones(N, J)
beta = 0.95
tau_mig = [(n == i && j == k) ? 0.0 : 1.0 for n in 1:N, j in 1:M, i in 1:N, k in 1:M]

alpha = rand(J)
alpha = alpha ./ sum(alpha)

nu = 1.0
T = 10

w_temp, pi_temp, X_temp, status_temp = solve_temporary_equilibrium(L_0)
println("[Section D reset] baseline temporary-equilibrium solver status: ", status_temp)

mu_stationary = recompute_mu_stationary();

[Section D reset] baseline temporary-equilibrium solver status: LOCALLY_SOLVED


### D.1 Baseline & One-Off Scenarios

**Baseline — no shock**

Same idea as Scenario 1 above (pure convergence to the baseline steady state), but at this section's default short horizon ($t=1,\dots,3$) with a period-by-period summary instead of a start/end snapshot.

In [32]:
run_simple_scenario("Baseline: N=2, J=3, no shock");
    # no A_shocks/kappa_shocks passed, so Tsim/t_start/t_end all take run_simple_scenario's
    # defaults (Tsim=3, t_start=1, t_end=3)

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 5.798724167505043e-9)

Baseline: N=2, J=3, no shock
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6862          0.2227          0.4962          0.5959          
  Region 2:  2.6862          0.6429          0.4613          0.2087          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.94            0.8722          
  Region 2:  0.8089          0.9538          1.0758          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2801          -0.0589         -0.1041         -0.118          
  Region 2:  0.2801          -0.1243         -

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.6861911601074766 0.2227032909381655 0.4961637822542615 0.5958656217163272; 2.6861911601074766 0.6428735150706475 0.4612886160595499 0.20872285374609484], [2.9662412155929623 0.16377253519009327 0.39202166640179087 0.4778538739195183; 2.9662412155929623 0.5186219744141325 0.3617605111269736 0.1534870077615669], [3.0164634450771954 0.15684480258898967 0.3727726792535548 0.45384397340312127; 3.0164634450771954 0.49237789472740146 0.34417296584933976 0.14706079402320182], [3.025620057305025 0.15580523118392237 0.3693226317067639 0.4492402312354722; 3.025620057305025 0.4871890817849043 0.3410993635230986 0.14610334595578792]], mu_path = [[0.585759890268867 0.32730289549070674 0.3104105379489311 0.30460612255311764; 0.21548902109275617 0.30193175345442613 0.3124842782338563 0.3282066610648155;;; 0.21548902109275617 0.32730289549070674 0.3104105379489311 0.30460612255311764; 0.585759890268867 0.30193175345442613 0.3124842782338563 0.3282066610

**Region 1, Sector 1 productivity +50% (one-time, period 1)**

The period-by-period path of a single-market productivity shock -- same idea as Scenario 2 above, but a bigger shock (+50%) and reported period-by-period over $t=1,\dots,3$ instead of start/end only.

In [33]:
run_simple_scenario("Region 1, Sector 1 productivity +50%";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=1)]);
    # ramp=1: one-time jump in period 1

solve_transition_path_hat converged after 23 outer iterations (max u_dot change = 5.217704046600602e-9)

Region 1, Sector 1 productivity +50%
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6593          0.28            0.5001          0.6005          
  Region 2:  2.6593          0.6255          0.4649          0.2104          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.7433          0.6897          
  Region 2:  0.6156          0.7542          0.8507          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2813          -0.0698         -0.1031         -0.1167         
  Region 2:  0.2813          -0.1194  

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.659277517235126 0.2800111748478351 0.5000783356817019 0.6005462077237956; 2.659277517235126 0.6254737548800344 0.4649352660198728 0.21040022637650657], [2.94058882165191 0.21022267160077485 0.39693967591177864 0.4838473954636811; 2.94058882165191 0.5061081456589706 0.3662985321405575 0.1554059359204172], [2.9913341328947203 0.2010116892327026 0.3777293689838531 0.4598990372831275; 2.9913341328947203 0.48097416934792164 0.34874041250416865 0.14897705685878615], [3.0006190251724756 0.1995773034000285 0.37425077494102377 0.4552609147372981; 3.0006190251724756 0.47602351443232827 0.34564028462893326 0.14800915751543828]], mu_path = [[0.5821479585722219 0.32046997192140275 0.30713181262581796 0.30134413674735955; 0.21416026567864496 0.2999361653931972 0.3091999187369216 0.3248872875595606;;; 0.21416026567864496 0.32046997192140275 0.30713181262581796 0.30134413674735955; 0.5821479585722219 0.2999361653931972 0.3091999187369216 0.324887287559

**Region 1 ↔ 2 trade costs double, every sector (one-time, period 1)**

A broad, symmetric trade-cost increase across both directions and all sectors between the two regions, with no productivity shock.

In [34]:
run_simple_scenario("Region 1 <-> 2 trade costs double";
    kappa_shocks=[(n=n_, j=j, i=i_, factor=2.0, ramp=1) for (n_,i_) in [(1,2),(2,1)] for j in 1:J]);
    # one spec per (destination n_, origin i_, sector j) generated by the comprehension below

solve_transition_path_hat converged after 27 outer iterations (max u_dot change = 5.013182757807044e-9)

Region 1 <-> 2 trade costs double
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.8478          0.2429          0.4138          0.488           
  Region 2:  2.8478          0.5269          0.4039          0.2288          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.7062          0.642           
  Region 2:  0.5975          0.7574          1.0853          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2865          -0.068          -0.1024         -0.1156         
  Region 2:  0.2865          -0.1226     

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.847804079682256 0.24287595414843632 0.41381223569372233 0.4880474103454125; 2.847804079682256 0.5268960437315049 0.4039327208088903 0.2288274759075215], [3.13427465374775 0.1749203627666665 0.3114344763750119 0.37249153414764247; 3.13427465374775 0.4042860279149978 0.3035996029007757 0.16471868839940734], [3.180674653206183 0.16643554057647234 0.29463432805291534 0.3518434764271762; 3.180674653206183 0.38166476684985695 0.28728398855590587 0.15678859312530843], [3.188259256550566 0.16521593487234873 0.2919207798451952 0.34832163583423353; 3.188259256550566 0.37769490027872316 0.2846678468687081 0.15566038919966202]], mu_path = [[0.6071900405880923 0.3454191646654516 0.3341600337543251 0.329456823514553; 0.22337273281641276 0.3270389925920823 0.3347943004420966 0.34637199130924234;;; 0.22337273281641276 0.3454191646654516 0.3341600337543251 0.329456823514553; 0.6071900405880923 0.3270389925920823 0.3347943004420966 0.34637199130924234;;;

**Migration cost = 0 everywhere, no productivity/trade-cost shock**

Tests how the labor distribution evolves under frictionless mobility alone (no fundamentals shock) -- isolates the pure migration-cost effect from any productivity/trade response.

`tau_mig` only enters via `mu_stationary` (`stationary_V`/`migration_shares`) -- the production side (`w_temp`, `pi_temp`, `L_0`, `A_0`, `kappa_0`) never reads it -- so this uses `with_taumig_override` (Scenario Infrastructure) to override `tau_mig`/`mu_stationary` just for this call, then restore both.

In [35]:
with_taumig_override(0.0) do
    run_simple_scenario("Migration cost = 0 everywhere (frictionless mobility)")
end

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 5.227693389286969e-9)

Migration cost = 0 everywhere (frictionless mobility)
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.4228          0.2791          0.5953          0.7064          
  Region 2:  2.4228          0.7561          0.5561          0.2613          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9369          0.877           
  Region 2:  0.8193          0.9497          1.07            

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  -0.0            0.0             -0.0            -0.0            
  Region 2:  -0.0    

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.4228219284861257 0.2791227041913111 0.5952664366943693 0.7064262094543092; 2.4228219284861257 0.7560779240849416 0.5561454678519685 0.261317400750849], [2.4228219284704897 0.27912270420374174 0.595266436690528 0.7064262094268223; 2.4228219284704897 0.7560779240750767 0.5561454678483794 0.26131740081447397], [2.4228219284704906 0.2791227042037415 0.5952664366905283 0.7064262094268224; 2.4228219284704906 0.7560779240750767 0.5561454678483797 0.261317400814474], [2.422821928470491 0.2791227042037416 0.5952664366905284 0.7064262094268228; 2.422821928470491 0.7560779240750769 0.5561454678483799 0.26131740081447413]], mu_path = [[0.30285274106076565 0.30285274106076565 0.30285274106076565 0.30285274106076565; 0.30285274106076565 0.30285274106076565 0.30285274106076565 0.30285274106076565;;; 0.30285274106076565 0.30285274106076565 0.30285274106076565 0.30285274106076565; 0.30285274106076565 0.30285274106076565 0.30285274106076565 0.30285274106

**Home-production value b -90% in every region, no other shock**

Tests how much a large drop in the non-employment outside option (home production) reallocates labor toward employment, holding everything else fixed.

`b` only enters via `flow_utility_mkt_at` → `stationary_V` → `migration_shares` (the baseline `mu_stationary`); `solve_temporary_equilibrium`/`solve_temp_eq_hat` never use it. `b` and `mu_stationary` are overridden just for this call, via `with_b_override`, then restored.

In [36]:
with_b_override(0.1) do
    run_simple_scenario("Home production value b -90% (all regions)")
end

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 6.072278901925188e-9)

Home production value b -90% (all regions)
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.5305          0.5462          1.3096          1.6051          
  Region 2:  0.5305          1.7571          1.2074          0.5136          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9468          0.8648          
  Region 2:  0.7916          0.9623          1.0862          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  -0.0669         -0.0583         0.0339          0.0852          
  Region 2:  -0.0669         0.1

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [0.5305035058240918 0.5462084939018566 1.3095856398139092 1.6051271173964157; 0.5305035058240918 1.757146295973928 1.207356821985767 0.5135686192799399], [0.46362602276969683 0.4879233342608719 1.3434551158865078 1.6903306335006794; 0.46362602276969683 1.8705404640637318 1.2216301131157699 0.45886829363304554], [0.4542848683603307 0.4800570580295651 1.347302554013092 1.7023500533134366; 0.4542848683603307 1.8877074292250766 1.2218014190745197 0.4522117496236485], [0.45300874972971206 0.47887422519992573 1.3472758743611093 1.7043690849595474; 0.45300874972971206 1.891231404991713 1.2210246620946628 0.4512072489336177]], mu_path = [[0.15854413908411394 0.0581491865526827 0.05056095065881397 0.04806623529550397; 0.058325129287271245 0.046865378295643675 0.051476647656602224 0.05851583899346011;;; 0.058325129287271245 0.0581491865526827 0.05056095065881397 0.04806623529550397; 0.15854413908411394 0.046865378295643675 0.051476647656602224 0.058

### D.2 Mobility Cost (tau_mig) Sensitivity

**Control — tau_mig=1 (baseline), A[1,1] +50% ramped over t=1-5, T=50**

The control case for the `tau_mig` sweep below -- same shock (region 1, sector 1 productivity +50%, ramped over periods 1-5, flat thereafter) at the current baseline `tau_mig` (=1 off-diagonal, as set by the reset cell above).

`tau_mig` only enters through `mu_stationary`, so this control needs no override -- it already reflects whatever `tau_mig` the reset cell left in place.

In [37]:
run_simple_scenario("Control: tau_mig=1 (baseline), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:tau_mig, swept_value=1.0);
    # A_shocks as in the +50% cell above, but ramped over 5 periods instead of a one-time jump

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 7.318309291903802e-9)

Control: tau_mig=1 (baseline), A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6806          0.2339          0.497           0.5969          
  Region 2:  2.6806          0.6399          0.4621          0.2091          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.8959          0.8312          
  Region 2:  0.7663          0.909           1.0253          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2755          -0.0524         -0.103          -0.1166         
  Region

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.6805618838834824 0.23389545569929932 0.4970114502367114 0.5968909981141554; 2.6805618838834824 0.6399381515767806 0.46207356846241576 0.20906660814367264], [2.956034665957426 0.1815194191239285 0.3939723360315545 0.4802479205510403; 2.956034665957426 0.5144131915427361 0.3635539058652246 0.1542238949706644], [3.000762057910918 0.1830981374267882 0.37595458837372525 0.45776063064314576; 3.000762057910918 0.4863081084832499 0.3470958586000278 0.14825856065122733], [3.0039791634721746 0.19147522467386627 0.37377533598627527 0.4548822298090519; 3.0039791634721746 0.4789916957199718 0.34517159078258874 0.1477455960838986], [2.999977075888226 0.20082075831023793 0.3743788337213905 0.45557357865992854; 2.999977075888226 0.475481367107655 0.3457468688936376 0.14804444153070054], [2.999415451138594 0.20136254070021298 0.3745702092705884 0.45580898926546964; 2.999415451138594 0.4753898807011527 0.3459229267712481 0.14811455101414137], [2.99932184

**Low — tau_mig=0 (frictionless), same shock as the Control above**

Tests the same productivity shock under frictionless mobility -- compare against the Control's convergence lag to see how much migration frictions slow the endogenous response.

Uses `with_taumig_override`, same as Scenario 9 above.

In [38]:
with_taumig_override(0.0) do
    run_simple_scenario("Low tau_mig=0 (frictionless), A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:tau_mig, swept_value=0.0)
end

solve_transition_path_hat converged after 23 outer iterations (max u_dot change = 5.102911426746459e-9)

Low tau_mig=0 (frictionless), A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.4172          0.2924          0.5958          0.7071          
  Region 2:  2.4172          0.7521          0.5566          0.2616          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.893           0.8359          
  Region 2:  0.7758          0.9052          1.0198          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  -0.0059         0.0138          0.0006          0.0008          
  Region 

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.4172356659400784 0.2924001529800581 0.5958017660729017 0.7070615059537487; 2.4172356659400784 0.752067221375972 0.5566456153539994 0.261552406383164], [2.411350485740033 0.30617982280643796 0.5964467255069708 0.707826904784591; 2.411350485740033 0.7477618487376704 0.5572481879905696 0.2618355386936928], [2.405153157363608 0.32046375864972626 0.5972130051952498 0.7087362791792492; 2.405153157363608 0.7431446049455661 0.5579641077023001 0.2621719296006929], [2.398630350293594 0.33525170247652736 0.5981129811926095 0.7098043162684796; 2.398630350293594 0.7381983501245375 0.5588049371885169 0.26256701216213996], [2.391768710579388 0.3505408748479368 0.5991596697313165 0.7110464629296033; 2.391768710579388 0.7329062336747596 0.5597828372534207 0.26302650040418535], [2.3917687105681265 0.35054087491604896 0.5991596697284954 0.7110464629262552; 2.3917687105681265 0.7329062336392125 0.559782837250785 0.2630265004029469], [2.391768710568127 0.35

**High — tau_mig=100000 (effectively no mobility), same shock as the Control above**

Tests the same productivity shock under near-zero mobility -- the other extreme from Low, to bracket how much migration frictions can slow the endogenous response.

Uses `with_taumig_override`, same as Scenario 10 above.

In [39]:
with_taumig_override(100000.0) do
    run_simple_scenario("High tau_mig=100000 (effectively no mobility), A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:tau_mig, swept_value=100000.0)
end

solve_transition_path_hat converged after 26 outer iterations (max u_dot change = 8.529449813110546e-9)

High tau_mig=100000 (effectively no mobility), A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             1.0             1.0             1.0             
  Region 2:  1.0             1.0             1.0             1.0             

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9512          0.9512          
  Region 2:  0.9372          0.9512          0.9512          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.0             0.0             0.0             0.0      

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]  …  [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0]], mu_path = [[1.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 1.0 0.0 0.0 0.0;;;; 0.0 1.0 0.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 0.0 1.0 0.0 0.0;;;; 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0; 0.0 0.0 1.0 0.0;;;; 0.0 0.0 0.0 1.0; 0.0 0.0 0.0 0.0;;

In [40]:
# Convergence-lag summary for the three tau_mig regimes above, queried from scenario_runs by the
# swept_param/swept_value tag each run was logged with (T==50 disambiguates this shock/horizon
# from Scenarios 9-10, which sweep tau_mig too but at T=200).
summary_cols = [:label, :swept_value, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
tau_summary = filter(row -> row.swept_param === :tau_mig && row.T == 50 && row.swept_value in (0.0, 1.0, 100000.0),
                      scenario_runs)
println(sort(tau_summary[:, summary_cols], :swept_value))

3×5 DataFrame
 Row │ label                              swept_value  shock_end_period  endog_converged_period  endog_convergence_lag 
     │ String                             Any          Int64             Union{Missing, Int64}   Union{Missing, Int64} 
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Low tau_mig=0 (frictionless), A[…  0.0                         5                       6                      1
   2 │ Control: tau_mig=1 (baseline), A…  1.0                         5                       7                      2
   3 │ High tau_mig=100000 (effectively…  100000.0                    5                       6                      1


**tau_mig sweep at intermediate values (0.1, 5, 20, 100, 1000)**

Tests whether `endog_convergence_lag` is non-monotonic in mobility frictions -- near-zero lag is expected at both `tau_mig=0` (migration reoptimizes essentially instantly) and `tau_mig=100000` (migration is frozen, so there's nothing left to lag); the interesting question is whether it's larger somewhere in between. `tau_mig`=0, 1, and 100000 were already run above (Low/Control/High); this fills in the gap. Same shock as before (region 1, sector 1 productivity +50%, ramped over t=1-5), T=50.

In [41]:
intermediate_tau_values = [0.1, 5.0, 20.0, 100.0, 1000.0]  # off-diagonal tau_mig values to sweep
for cost_value in intermediate_tau_values
    with_taumig_override(cost_value) do
        run_simple_scenario("tau_mig=$cost_value, A[1,1] +50% ramped over t=1-5, T=50";
            A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:tau_mig, swept_value=cost_value)
    end
end

solve_transition_path_hat converged after 23 outer iterations (max u_dot change = 4.98406538262941e-9)

tau_mig=0.1, A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.4385          0.2874          0.5879          0.6984          
  Region 2:  2.4385          0.7434          0.549           0.257           

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.8932          0.8354          
  Region 2:  0.775           0.9055          1.0203          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.0166          0.0082          -0.0076         -0.0084         
  Region 2:  0.0166        

In [42]:
# Full tau_mig scan summary: every Section D tau_mig-swept run (Control/Low/High plus the
# intermediate sweep, all at the same shock and T=50), queried from scenario_runs by
# swept_param/swept_value and sorted ascending -- read endog_convergence_lag down this table to
# see whether it peaks away from the two boundaries (expected) or is flat/near-zero throughout.
summary_cols = [:label, :swept_value, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
tau_scan = filter(row -> row.swept_param === :tau_mig && row.T == 50, scenario_runs)
tau_scan = sort(tau_scan[:, summary_cols], :swept_value)
rename!(tau_scan, :swept_value => :tau_mig_offdiag)
println(tau_scan)

8×5 DataFrame
 Row │ label                              tau_mig_offdiag  shock_end_period  endog_converged_period  endog_convergence_lag 
     │ String                             Any              Int64             Union{Missing, Int64}   Union{Missing, Int64} 
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Low tau_mig=0 (frictionless), A[…  0.0                             5                       6                      1
   2 │ tau_mig=0.1, A[1,1] +50% ramped …  0.1                             5                       6                      1
   3 │ Control: tau_mig=1 (baseline), A…  1.0                             5                       7                      2
   4 │ tau_mig=5.0, A[1,1] +50% ramped …  5.0                             5                      15                     10
   5 │ tau_mig=20.0, A[1,1] +50% ramped…  20.0                            5                 missing                missing

### D.3 Discount Factor (beta) Sensitivity

**Control — beta=0.95 (baseline), A[1,1] +50% ramped over t=1-5, T=50**

The control case for the `beta` (discount factor) sweep below, using the same shock as the `tau_mig` sweep above.

Unlike `tau_mig`/`b`, `beta` enters the dynamics directly (`migration_shares_next` and the backward `u_dot` step in `solve_transition_path_hat` both read the global `beta` every period, not just the baseline `mu_stationary`) -- but the control still needs no override, since it already reflects whatever `beta` the reset cell left in place.

In [43]:
run_simple_scenario("Control: beta=0.95 (baseline), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:beta, swept_value=0.95);

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 7.318309291903802e-9)

Control: beta=0.95 (baseline), A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6806          0.2339          0.497           0.5969          
  Region 2:  2.6806          0.6399          0.4621          0.2091          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.8959          0.8312          
  Region 2:  0.7663          0.909           1.0253          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2755          -0.0524         -0.103          -0.1166         
  Region

**Low — beta=0.5 (households barely value the future), same shock as the Control above**

Tests how impatient households change the speed/shape of the migration response.

In [44]:
with_beta_override(0.5) do
    run_simple_scenario("Low beta=0.5, A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:beta, swept_value=0.5)
end

solve_transition_path_hat converged after 23 outer iterations (max u_dot change = 6.140651320762913e-9)

Low beta=0.5, A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.7868          0.5326          0.8004          0.8942          
  Region 2:  1.7868          0.9351          0.7686          0.4955          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9227          0.888           
  Region 2:  0.8374          0.9302          0.9992          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.1569          -0.05           -0.0529         -0.0469         
  Region 2:  0.1569      

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.7867789381082284 0.5325944445436724 0.8004359096811141 0.8941646302730217; 1.7867789381082284 0.9351317132382511 0.7685669150236891 0.49554851102379494], [1.9436935134216393 0.4826338056231956 0.7475708953806216 0.8473076007018404; 1.9436935134216393 0.8864583745505422 0.7136366327770288 0.43500566412349334], [1.9726630537893803 0.4867291533097156 0.7357538275333507 0.8348808955694189; 1.9726630537893803 0.8685830404939981 0.7019959795378379 0.42673099597691955], [1.974466138170078 0.499031153706252 0.7336280542246494 0.8324415524424433; 1.974466138170078 0.8603303676413531 0.6999693559521843 0.4256672396929626], [1.9706717254855683 0.512274453759254 0.7336480207558939 0.8324301441127349; 1.9706717254855683 0.8545461429520632 0.699998941341758 0.4257588461071602], [1.9699903661171854 0.513742360303537 0.733791059347427 0.8325862575910221; 1.9699903661171854 0.8539085026866629 0.7001374649978713 0.4258536228391096], [1.969858444798292 0.

**High — beta=0.99 (households very patient/forward-looking), same shock as the Control above**

The other extreme from Low -- how forward-looking households change the migration response, bracketing the `beta` sweep.

In [45]:
with_beta_override(0.99) do
    run_simple_scenario("High beta=0.99, A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:beta, swept_value=0.99)
end

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 7.161154336188247e-9)

High beta=0.99, A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.7569          0.2148          0.4695          0.5666          
  Region 2:  2.7569          0.6084          0.4354          0.1915          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.8934          0.8267          
  Region 2:  0.761           0.907           1.027           

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2773          -0.0508         -0.1042         -0.1193         
  Region 2:  0.2773    

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [2.7569012433633344 0.21482837364759194 0.4694698234219524 0.5666009522105461; 2.7569012433633344 0.6084343936285611 0.43537681596674577 0.1914871543979333], [3.034209650687194 0.16407120872011827 0.36529458754716065 0.44731952155023774; 3.034209650687194 0.4798188054865578 0.3362651900225001 0.13881138529903522], [3.0776840780484203 0.16554897330150914 0.34778315134942184 0.42530345916426765; 3.0776840780484203 0.452393585510865 0.3203385251241815 0.1332641494529126], [3.080551664203747 0.17342906826161217 0.34579992842731316 0.4226655365112705; 3.080551664203747 0.4455836748757579 0.3185964471062255 0.13282201641032484], [3.076606160208053 0.18222449868122167 0.34644321182228743 0.42341380079367374; 3.076606160208053 0.4423799932137261 0.3192052693830673 0.133120905689915], [3.0760708408444666 0.18270103850943528 0.3466289169540534 0.4236441423784741; 3.0760708408444666 0.44232315537832273 0.31937531071639824 0.13318575437437974], [3.075

In [46]:
# Convergence-lag summary for the three beta regimes above, queried from scenario_runs by
# swept_param/swept_value (same style as the tau_mig summary cell above).
summary_cols = [:label, :swept_value, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
beta_summary = filter(row -> row.swept_param === :beta && row.T == 50, scenario_runs)
println(sort(beta_summary[:, summary_cols], :swept_value))

3×5 DataFrame
 Row │ label                              swept_value  shock_end_period  endog_converged_period  endog_convergence_lag 
     │ String                             Any          Int64             Union{Missing, Int64}   Union{Missing, Int64} 
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Low beta=0.5, A[1,1] +50% ramped…  0.5                         5                       7                      2
   2 │ Control: beta=0.95 (baseline), A…  0.95                        5                       7                      2
   3 │ High beta=0.99, A[1,1] +50% ramp…  0.99                        5                       7                      2


### D.4 Migration Taste Dispersion (nu) Sensitivity

**Control — nu=1.0 (baseline), A[1,1] +50% ramped over t=1-5, T=50**

The control case for the `nu` (migration taste-shock dispersion) sweep below, using the same shock as the `tau_mig` and `beta` sweeps above.

Like `beta`, `nu` enters the dynamics directly (`migration_shares_next` and the backward `u_dot` step both read the global `nu` every period) -- but the control still needs no override, since it already reflects whatever `nu` the reset cell left in place.

In [47]:
run_simple_scenario("Same: nu=1.0 (baseline), A[1,1] +50% ramped over t=1-5, T=50";
    A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:nu, swept_value=1.0);

solve_transition_path_hat converged after 22 outer iterations (max u_dot change = 7.318309291903802e-9)

Same: nu=1.0 (baseline), A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.6806          0.2339          0.497           0.5969          
  Region 2:  2.6806          0.6399          0.4621          0.2091          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.8959          0.8312          
  Region 2:  0.7663          0.909           1.0253          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.2755          -0.0524         -0.103          -0.1166         
  Region 2:  0

**Low — nu=0.1 (migration dominated by utility differences: little idiosyncratic noise, very elastic/responsive migration), same shock as the Control above**

Tests how sharply responsive (low-noise) migration changes the speed/shape of the reallocation.

In [48]:
with_nu_override(0.1) do
    run_simple_scenario("Low nu=0.1, A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:nu, swept_value=0.1)
end

solve_transition_path_hat: reached max_outer=100 without converging (max u_dot change = 845.6693055410065)

Low nu=0.1, A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.0             0.0             0.0             7.9657          
  Region 2:  0.0             0.0             0.0             0.0343          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             653.9937        0.0             
  Region 2:  3.6357          653.9806        0.0             

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  -0.0            -0.0            -0.0            0.0332          
  Region 2:  -0.0       

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [5.926000417553575e-23 4.223506059946039e-14 1.3440388488651704e-25 7.965675658004512; 5.926000417553575e-23 1.2515880207155986e-43 4.43577022875236e-26 0.034324341995445894], [1.6562356260502517e-48 7.693118386130957e-21 2.1761059233364688e-30 7.998886847316534; 1.6562356260502517e-48 4.2305856304476586e-107 7.327265849071989e-31 0.001113152683466255], [7.104891418677339e-50 2.7342417105972984e-22 1.0758790881736677e-27 7.999963899460855; 7.104891418677339e-50 1.3584330386234738e-112 6.664177511806144e-29 3.610053914465061e-5], [2.261436476297446e-63 7.970967954018494 0.02733870732455407 8.447096410389921e-34; 2.261436476297446e-63 3.600576692798334e-147 0.0016933386569524943 6.396878309736351e-44], [1.3173915667836059e-79 7.971518450035941 0.027044063013098706 4.401430485907512e-61; 1.3173915667836059e-79 4.132360931369902e-170 0.001437486950961989 6.677978076246826e-67], [1.3743162256672055e-79 7.971944579585615 0.026802743472861996 4.5

**High — nu=5.0 (migration dominated by idiosyncratic taste: sluggish, barely responsive to utility gaps), same shock as the Control above**

The other extreme from Low -- how noisy, unresponsive migration changes the reallocation, bracketing the `nu` sweep.

In [49]:
with_nu_override(5.0) do
    run_simple_scenario("High nu=5.0, A[1,1] +50% ramped over t=1-5, T=50";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=50, swept_param=:nu, swept_value=5.0)
end

solve_transition_path_hat converged after 24 outer iterations (max u_dot change = 9.235351816627713e-9)

High nu=5.0, A[1,1] +50% ramped over t=1-5, T=50
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  1.2678          0.8086          0.9448          0.9888          
  Region 2:  1.2678          1.0092          0.9298          0.7832          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9407          0.9287          
  Region 2:  0.8966          0.9437          0.973           

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.0062          0.0028          -0.0024         -0.0014         
  Region 2:  0.0062       

(L_path = [[1.0 1.0 1.0 1.0; 1.0 1.0 1.0 1.0], [1.2678460835553045 0.8085781383001029 0.9447836765803413 0.9887761872750442; 1.2678460835553045 1.0091910676359712 0.9298254593643255 0.7831533037336054], [1.27409283775605 0.811328976797728 0.9424184701816875 0.9873936709077088; 1.27409283775605 1.005680093409461 0.9271358430544506 0.7778572701368637], [1.2724303946608975 0.8188680764532796 0.9421297788187173 0.9871151261222811; 1.2724303946608975 1.0026239714212937 0.9268440083325047 0.7775582495301283], [1.2705014165340658 0.8265069615567057 0.9419585594860064 0.9869358364062616; 1.2705014165340658 0.9995031688328286 0.9266755341814547 0.7774171064686114], [1.2685879092587393 0.8339676259616715 0.9418347607133776 0.9868059285280179; 1.2685879092587393 0.9963465096239345 0.9265538095596151 0.7773155470959049], [1.268532915796926 0.834137289043384 0.9418383103435635 0.9868094958597569; 1.268532915796926 0.9962727851539107 0.9265573518245923 0.7773189361809407], [1.2685313119206583 0.8341

In [50]:
# Convergence-lag summary for the three nu regimes above, queried from scenario_runs by
# swept_param/swept_value (same style as the tau_mig and beta summary cells above).
summary_cols = [:label, :swept_value, :shock_end_period, :endog_converged_period, :endog_convergence_lag]
nu_summary = filter(row -> row.swept_param === :nu && row.T == 50, scenario_runs)
println(sort(nu_summary[:, summary_cols], :swept_value))

3×5 DataFrame
 Row │ label                              swept_value  shock_end_period  endog_converged_period  endog_convergence_lag 
     │ String                             Any          Int64             Union{Missing, Int64}   Union{Missing, Int64} 
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ Low nu=0.1, A[1,1] +50% ramped o…  0.1                         5                 missing                missing 
   2 │ Same: nu=1.0 (baseline), A[1,1] …  1.0                         5                       7                      2
   3 │ High nu=5.0, A[1,1] +50% ramped …  5.0                         5                       6                      1


### D.5 Solver-Tuning Diagnostics

**Diagnostic — tau_mig=20, T=200, damp=0.2**

Reruns the tau_mig=20 case with a longer horizon and lower damping, to see whether the endogenous path (wages, labor, utility) converges given enough time and outer iterations. `tau_mig=20` sits in the "interior" friction regime, where migration reallocates gradually over many periods rather than snapping instantly (`tau_mig=0`) or staying frozen (`tau_mig=100000`).

In [51]:
result_tau20 = with_taumig_override(20.0) do
    run_simple_scenario("tau_mig=20.0 diagnostic rerun, A[1,1] +50% ramped over t=1-5, T=200, damp=0.2";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=200, max_outer=500, damp=0.2)
        # Tsim=200: simulation horizon; max_outer=500: outer-loop iteration budget; damp=0.2: how
        # much of each new outer-loop guess to blend in (see solve_transition_path_hat's damp comment)
end

println("\nouter loop converged: ", result_tau20.converged, "   outer_used: ", result_tau20.outer_used)
println("Per-period endogenous deviation from steady state (max|.-1| across markets):")
Tu = length(result_tau20.u_dot_path)
for p in 1:Tu
    # dev_w/dev_L/dev_u: the largest deviation from "no further change" (|.-1|) across every
    # market at period p, for wages, labor, and utility respectively -- all three should shrink
    # toward 0 as the endogenous path settles into its new steady state
    dev_w = maximum(abs.(result_tau20.w_path[p+1] ./ result_tau20.w_path[p] .- 1))
    dev_L = maximum(abs.(result_tau20.L_path[p+1] ./ result_tau20.L_path[p] .- 1))
    dev_u = maximum(abs.(result_tau20.u_dot_path[p] .- 1))
    println("  t=$p   dev_w=$(round(dev_w,digits=6))   dev_L=$(round(dev_L,digits=6))   dev_u=$(round(dev_u,digits=6))")
end

solve_transition_path_hat: reached max_outer=500 without converging (max u_dot change = 6278.065585374409)

tau_mig=20.0 diagnostic rerun, A[1,1] +50% ramped over t=1-5, T=200, damp=0.2
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.3479          0.3111          0.6185          0.7264          
  Region 2:  2.3479          0.788           0.5804          0.2799          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.904           0.8533          
  Region 2:  0.7782          0.9156          1.0326          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.6134          -0.2001         -0.2203         -0.2034    

┌ Warning: JSON-RPC endpoint has been blocked writing to its peer; outgoing messages are queueing up and will not be delivered until the peer resumes reading
│   blocked_seconds = 308.3
│   queued_messages = 1
└ @ VSCodeServer.JSONRPC /Users/jonathantong/.vscode/extensions/julialang.language-julia-1.243.2/scripts/packages/JSONRPC/src/core.jl:414


false   outer_used: 500
Per-period endogenous deviation from steady state (max|.-1| across markets):
  t=1   dev_w=0.221775   dev_L=1.347854   dev_u=0.252995
  t=2   dev_w=0.194936   dev_L=0.679177   dev_u=0.208218
  t=3   dev_w=0.170563   dev_L=0.640186   dev_u=0.173908
  t=4   dev_w=0.149659   dev_L=0.603411   dev_u=0.149728
  t=5   dev_w=0.133499   dev_L=0.569257   dev_u=0.133761
  t=6   dev_w=0.065474   dev_L=0.538063   dev_u=0.123898
  t=7   dev_w=0.056754   dev_L=0.509728   dev_u=0.119495
  t=8   dev_w=0.049169   dev_L=0.484264   dev_u=0.118098
  t=9   dev_w=0.042601   dev_L=0.461571   dev_u=0.118864
  t=10   dev_w=0.036929   dev_L=0.441476   dev_u=0.121203
  t=11   dev_w=0.032037   dev_L=0.423765   dev_u=0.124702
  t=12   dev_w=0.027816   dev_L=0.408209   dev_u=0.129078
  t=13   dev_w=0.024184   dev_L=0.394573   dev_u=0.134136
  t=14   dev_w=0.021043   dev_L=0.382642   dev_u=0.139747
  t=15   dev_w=0.018329   dev_L=0.372212   dev_u=0.145824
  t=16   dev_w=0.015981   dev_L=0.3631

**Diagnostic — tau_mig=12, T=100, damp=0.2**

Same diagnostic as tau_mig=20 above, at `tau_mig=12`, with a larger outer-iteration budget (`max_outer=1000`).

In [52]:
result_tau12 = with_taumig_override(12.0) do
    run_simple_scenario("tau_mig=12.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=100, max_outer=1000, damp=0.2)
        # Tsim=100, max_outer=1000, damp=0.2: solver settings for this rerun
end

println("\nouter loop converged: ", result_tau12.converged, "   outer_used: ", result_tau12.outer_used)
println("Per-period endogenous deviation from steady state (max|.-1| across markets):")
Tu = length(result_tau12.u_dot_path)
for p in 1:Tu
    # dev_w/dev_L/dev_u: see the tau_mig=20 diagnostic above for what these measure
    dev_w = maximum(abs.(result_tau12.w_path[p+1] ./ result_tau12.w_path[p] .- 1))
    dev_L = maximum(abs.(result_tau12.L_path[p+1] ./ result_tau12.L_path[p] .- 1))
    dev_u = maximum(abs.(result_tau12.u_dot_path[p] .- 1))
    println("  t=$p   dev_w=$(round(dev_w,digits=6))   dev_L=$(round(dev_L,digits=6))   dev_u=$(round(dev_u,digits=6))")
end

solve_transition_path_hat converged after 80 outer iterations (max u_dot change = 9.054089034066237e-9)

tau_mig=12.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.872           0.2101          0.423           0.4997          
  Region 2:  2.872           0.5387          0.396           0.1886          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9011          0.8465          
  Region 2:  0.7763          0.9131          1.0287          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.6444          -0.1592         -0.2364         -0.2536         
  R

**Diagnostic — tau_mig=17, T=100, damp=0.2**

Same diagnostic, at an intermediate value `tau_mig=17`, between the `tau_mig=12` and `tau_mig=20` cases above.

In [53]:
result_tau17 = with_taumig_override(17.0) do
    run_simple_scenario("tau_mig=17.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2";
        A_shocks=[(n=1, j=1, factor=1.5, ramp=5)], Tsim=100, max_outer=1000, damp=0.2)
end

println("\nouter loop converged: ", result_tau17.converged, "   outer_used: ", result_tau17.outer_used)
println("Per-period endogenous deviation from steady state (max|.-1| across markets):")
Tu = length(result_tau17.u_dot_path)
for p in 1:Tu
    # dev_w/dev_L/dev_u: see the tau_mig=20 diagnostic above for what these measure
    dev_w = maximum(abs.(result_tau17.w_path[p+1] ./ result_tau17.w_path[p] .- 1))
    dev_L = maximum(abs.(result_tau17.L_path[p+1] ./ result_tau17.L_path[p] .- 1))
    dev_u = maximum(abs.(result_tau17.u_dot_path[p] .- 1))
    println("  t=$p   dev_w=$(round(dev_w,digits=6))   dev_L=$(round(dev_L,digits=6))   dev_u=$(round(dev_u,digits=6))")
end

solve_transition_path_hat converged after 385 outer iterations (max u_dot change = 8.17973222488888e-9)

tau_mig=17.0 diagnostic, A[1,1] +50% ramped over t=1-5, T=100, damp=0.2
-- t=1 --
Labor distribution
  mass of households in each region-market at t=1
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  2.5651          0.2686          0.5375          0.6331          
  Region 2:  2.5651          0.6855          0.5037          0.2414          

Wage change w_1 / w_temp
  nominal wage at t=1 relative to the pre-shock baseline
             Sector 1        Sector 2        Sector 3        
  Region 1:  1.0             0.9034          0.8505          
  Region 2:  0.7771          0.9152          1.0313          

Net migration flow (L_1 - L_0)
  should -> 0 as the labor distribution converges
             Non-employment  Sector 1        Sector 2        Sector 3        
  Region 1:  0.6518          -0.1857         -0.2365         -0.2369         
  R